# Fundamentação teórica do capítulo fotovoltaico

<details>
<summary>Descrição e método</summary>

SPGER0046 UFPB. Levantamento de livros, artigos, relatórios e fontes de método para as quatro seções e 19 subseções. Adaptação do notebook multibase WCNPS da pasta Artigo/Bibliometria. Esta versão não busca lacuna científica nem produz uma revisão sistemática automática. As saídas identificam DOI, autoria, ano, periódico ou editora, acesso, tópico e citações separadas por base e data. Livros e artigos não são classificados como se tivessem cobertura bibliométrica idêntica.

</details>

## 0 Preparação e local de trabalho

<details>
<summary>Descrição e método</summary>

O código pode executar em Jupyter ou Colab. Credenciais são lidas de variáveis de ambiente ou dos mesmos Secrets usados no WCNPS: OPENALEX_API_KEY, SCOPUS_API_KEY, WOS_API_KEY e IEEE_API_KEY. Nenhuma chave é exibida ou gravada. Sem as chaves institucionais, o levantamento continua com Crossref e OpenAlex e registra as bases não executadas. O notebook executado já permite consultar os resultados. NOVA_COLETA=False reutiliza o retrato documentado; True cria outra pasta de respostas e consulta novamente.

</details>

In [ ]:
from pathlib import Path
import sys,subprocess,importlib.util,importlib.metadata,json,io,zipfile,base64,hashlib
from datetime import datetime,timezone,date
requirements={"requests":"requests>=2.32,<3","pandas":"pandas>=2.2,<3","docx":"python-docx>=1.1,<2"}
missing=[requirement for module,requirement in requirements.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.run([sys.executable,"-m","pip","install",*missing],check=True)
import pandas as pd
from IPython.display import display,Markdown,HTML
display(HTML("<style>.text_cell_render,.jp-RenderedHTMLCommon{font:20px/1.65 Arial,sans-serif}summary{cursor:pointer}.dataframe{font-size:17px}td,th{padding:8px}</style>"))
BIB=next((p/"bibliografia" for p in [Path.cwd(),*Path.cwd().parents] if (p/"bibliografia/temas.json").exists()),None)
if BIB is None:BIB=Path.cwd()/"Projeto_FV_PVDAQ_2107_2024/bibliografia"
BIB.mkdir(parents=True,exist_ok=True)
support=base64.b64decode("UEsDBBQAAAAIAM27UF3XjT07CSYAAIF6AAASAAAAYmlibGlvZ3JhZmlhX2Z2LnB5zX3bctw4luC7voLteiBpUWlJtqu3Us2aUdmqCU/V2A7b1du1WdkMJomU2GKS2QSplEqjiP6H2R+o2IeOnoh6mp2Y99WfzJfsuQAgeMm0XNMdMQpbSoLAAXBuODjnAPno0aN3Yl2VaZNki1w49YUoK1FnSZw7lYjTrDh3ZFNdiZvAKcoa/l+JvL5xUlGLpM7KYrK397Yqr7JUVE6S1TGWSWi6irPCkWIdV3EtJs6LSqSiqLM4l05cCYLtQI2VWJXVjVMW+c1k70WcXIgU3sk1ABHSKcQVgM2KJG9SbPPHRsja+e7dtzIwTxcASVTSKSvn9O0r51LcyMneo0eP9pZVuXLWcX2RZwsnW63LqnbewiO/SGFYdbYS+o1+Dhz8/WNZCFUvWy4tAO+x1yIR/xTXMNaK6yRlkTRVBfObLJu6gfHr6h8ucKJvyzI/uxZJU5fVnnpzEUscWHBRr/LgD7IsglIGlQhoDE2RJWUqYEyxrq+mK/XzOi7SGHApnXWqy5ZltYImSVxG8aKo8SX+3dt7/+a7dy/O3ocz981aFKe5uHYD90VVSlmJJXx8n5TrRsKH//nmPfx+dXZ25vxunQMnuPO9s9cv37559frD+/C2bT51L+p6LadPnsTrbFJCcQzFk7I6f7Ipq0uAtee0PXRrJ6q4V1sNoltX5FJcZaKaJOXqCeC5BiQ/kSKukosnUo0a2uK4e53kcZVdIe9hSyiR0JU8kHVc1aJ6cnX0JC2TZgXgGII95xZSJoS4pkKEiU80aHhACGocADFLciHdu733Zy/enX2IXp/+01kPXW/enr0+/fbsdxEwafTN2fct1qfu+xdv3n733nrD04HfVll3gPhkXt7tfXj14duz6B9OP2C3MJujyZE7nc1cWQIanHUpM5RLgMIF56Jcibq6wYKmGL7/URRZfeHO5wHBOrZgZVUFWiEGETC1qYDaz4OZe56XiziHl2lWgYqgD8tlI7E+kAIkG3AOn7EVjcJ085S62dkBjrdaxolw6A11eAWEjhcIPS+bFP7GAhisxCHE9aqUa5BT+AxyucjSrL4x/T2zptXpgkuELJvKGoU1MOx2JWIYC7z9YxPnCDZwG3hX1aD46GlVAhJLrHEFFUCWVcfHija1WK0FaEfQFzg8GOUK8Iag1xdlXV6VOYBK4NX6ylmBgs7bofQeQReI3EBnasX5QqQl4lksc6ADj1tVeapmnuWg35FCjUSSyAtS+GNjYHTTWAwUxl+d5di2rDKgq0Zh/GO2apCDRmdj4DGkpxohVVxIixcRskCCrnOYoFMuYeWo4hsC2uGSAfEM4OO/IaafKjQCG6dYAxQ0KJwcsAALYZU0Wb0VkU4i8hYM45HnFrjAPaICTVfw+oqMlMkVLjfbsKnhPGf5KWDNBBCm+HMqFoWozm+cm0zkiFL1uIZlOdP9AI5oBQGk4rzFNhQpyM8U2XLQ5shD4hzo0OqB7WTXSNVglHpZiyRbZokZ4jKDNcU8ARuLCldeU2IPF80M0CVm1s8Ubew6FY7NvGeka8kEhFQVySpqJBrwAmQDu1lApxeruLokkQasOKjFlO4ZJe663IiqhyqmzaLJcjKpkMjnaBlRD9n6CucLCzcx0nkO4kNyyfA2MJByQ6Du9vZSsXRgtYkL79qf7jnwA/ZaUxXwZyKbhVe5P8h9aAvoRtti0hRCJvFaeOq9+5vZ77+c73/JVWRdeddoO7muDz8TeM7Wns/dFIC6Lb24s9/HBz8eHnwxV31ZJssE2wFifxSe+/rrb166gR6vPwFkQi3PjWWSZcjt5wUuaP4kFfTCn+SIPa8/lLTMts739x4t138H67WXXv8w8f8OatM6/c/wYfqDfOxDT925aujBMo/PZQiwXpmuuUcpEujIK4CvVL/ALI0ISwmTuMoqMH3P1fvAAkg1syVXnqqh0gO9gUWXgeEPGY/nZXmeo5WSx8bKhNWyIuNP11RwcAb6nend702JGonrRKxr54z+gEjqkbguzy5eQO04qSNtunmgN9Q8syWZ+lBgtcIXZQWWtkjDW62kp2C9pWh2Ovgh0MUSLXtoPslA80rPpxr6Jb4zFe9sggIbTf5QZoWn+pllc2qZYRMJeBGpfuUrIt2ABRZpe9LbqAlgoyVqCWwIQtqALEvYWRysq4xsj7YENh5ZIewiZEopG/gwbymFuwMZehtCOoEmrN/e+VTi4vsDUApgTNKL2dw0BWxSawcMdv40O5xrxOpn+Gdj4jVuP2iCYELBlD02RYJNUJfrLAmWAncfqZaHEHW4qhOqqrBbqG4iqh5yowo1G2yo0iiuQwUhAAEJQTTqrM4Ffohh1S4riR8RtyGOJNBzuRJFQ7W06RzVN2sq0Du/KCmbouZWPPQoS7FCU+X4p4yj9hPY4nVDXWVyUeBf3ZHmTiwD7Qp9cS3J3a/jc0HtNNEqfICVDIeganbgSSFwIDKczQP6TBhpH6sSzHfzBFwC1kx9Ex4aaVa4DS2L3hA4LxOae8jc4UJzWC5uIl3uKlY5kVUS6kKuqixMXaGMNQyUyihOEiGlftvqgkmzRr7xkHaoF1Ub+Ag6XJFSl2USjKebCPUEA1IvqBZqRZ/prMeOCOURRlgOr02/QBbNHCdaUj1WQy6/cXtCYXeOXZFgxiiUG7uZvMjWYEHM5n6nN+Y2QNoWaD0u1BODz/y6x5PqPZQCiRc3XNqdX8uxqnKWMijk2S7lwCLFhTxCVkSW7iCXKGE9azBWV0oQSoU/fnSVKreEw6rAJaaOkZGhKtfY1S+USQgzK1JgXVjnzUgWeqqLDChfbmc2JYYLrs1PZiwsmeodPdivipaI+BS187QQon6UaP/nn/5FcRgt2sw516TOVS/LrJI1oR8Qu9BEMUVzlNprNVGR2yLceiXahbjJc5YbryMfrMtdd+5rBa30gaoF9oeqODUg9kN36rj7m1n7dm633ia+L9+8asXXQGPp7C9zu4XSrKMzxTznGcgSs6AqWcarLL+horkxHFrx9DZDsVZFqGV10bjE6sborYG9sKgORpBpQTRK/EFSbXU3LuCZPDB7hfRgcXOgBb2V7iHWURZVyXfvvh30xOuTQbCq+er9V68NIlp5ZGO3J4ME0pK7vlxtdsjVZkyuWFA0ConnLZwql9RB0awWotolb2YFHVBDN9LrqsUA8NgHzUvuEEXvWxSNCaP2gX1UPGBVlavp6BqXTPurGSqNTrukBA34EsAyHWbTZ8SBHevGkqUWMuwBYnbiAGzm7w5ga8F8vXthQm0Ajy9hNwbSprbcD2FoWrFG+VhTxF6mCnFde941v/n7C/bvupYCVY3A7L2kNZcUpaqOtX2givKs+v0BGh5vEZT25jNGZHJntuozle36ijEBsKcr2bGSNMo65tF2FgGI202gAW/otUhx+vcPMHPijgnyemjPaG5D40T2zCAF7qMGDs/2QzvcnpXdly0sla3+GbBFM2J29DiMmSXR+pr4iyaVWJxiojqGXbRFtiBmQer6AYpSpytkRs9itj5aeLxbeabj7v5Ftu9fycTVKyasyGOWbHep/ETajwyp3kZ/a1XVZR+xc9k+XYtqt61rKUu1bERKtxNt7LURHUpkqLIt0dMG3bVubCeuJ2fQ4hIesES3ha3xjFrM1UKqHxlgNTON5x0iDWwYu+akEiC4iQDTyHspEhFL2Dy77LZZ5xmM/cQlzjZwfOOtATjEJvMwK2rPPFF18+RgnIq37cTK7H3xfot+nzP0LAYfgF70ybcgmgaqoy4p2y775brzfjkYKuS6QbifOqRB36ap9rft7SVgYUvnPcWDWSjRRxHBviKro8iTIl8GVVnWQQ4WZ/iharTrDH/w7QTfhhiF9fCTf0KFyfI8xBDoJC/jVHqeqfkEVtmyLpMyLydYAeiF8dSoRr1lWVPUQO3ptwCqxSqWY0B4CLjx39pWChRDMdq8OwaaN/5iqBiMDm/lVPsSmT1lQE4wO1aoPWV33C4vz9Ed0UNdvAmtQcGjy7XLprZfwOO6qaXb7m7Q+0YCqOEEup35AJguqwjGGZeuVYgDAZFcT1aXaVZ56xiD3JIoG4jrDLZb5SXTec+wA+oC4gTlicrjhcgD8tBL9gypkD0/yGa5zK5DkMWWVRJMBAj1aJ94rdP59OB/sd85mhzMQYIjV3WzDx/3qSt/31WEsndsBHJCYwY8Ty1NCLIB++Mcds82/bn+KKVbagN2JvEatt2pt80FF/L0ta+LwKJnvl6Hx4eHXaecHsfMtYvdea9vJY9tbXQHu/NgW3MbDagdDKdO/3pTgoVTooycOJjoAXwgQZlcxWms5krmweg0iAmMrtH8GtcYtKtJkcfFufC0opi5q/g6WzWrSFUB/uyRs+Njb3vjpJJQJ1LQmmVyG2Y8wfk+s6NmV/5jGFb9pSQNI3U0KAU1Um9QksoixbENRqLwpgc04ecIwx+DumgRcXXkluGs8Aep30JDFvZAqdXxah3qpJZJUW48ndcyaerEn2Sy5HQRbzhC/PnMeQkG2QLDpiK/gVGAIYbZQQaRlLMTOJsM1timdhKT4HOwgMUNg1w6Q4eiFaOdsJBtKtB+LGUkgGmzWkvvtsvFU5pRoPcEU8WQLnGkO2WWZEmY4u87EAfZVCKiSFP4dQyroT8+1V/M+GUu6hiD7MTiXNyVaBr0lm7bEEAcUL3t1FfxGGf27PiL4DmojeeHx/D/Kfx/Np8uQEtddhWaWuwNr7/jDyYcFLRmgI9pQtBgyFxqnmhpelDBn0RkAEdRpyZaTSyJ+0e/2SWlU+JEmQux9lZgsx0/fqxeBs8G6+inU2MZ5xcxIPU8Q0vSKB5+bXte+mrHLFwElZcujpio3jDqG5J9lJbSXqZ4GFR3phhxfkKj0oWwBbOMzwktWD4tVbQBIqsi0pARldRXaN65bMVhaTsDsCpCY16QGtPC4PonebbKbMWUFRKs7gaJHucRbgbUQO1Vgduzjb4rF4xiWMiL0LMOYc1GII0HSvBnmeXh0sXAZ2TvelBRTW/NqB9RBTSSH83vDg6P4B+QZGeL+qIqm/MLKodGQSYjlETYUoh0ukTpd/ts28f19IGD4wbR336MahG6dTnRzJ0S6TAzIsdUBxwvpUEckP9v2hJdRwIiNdSW7Nt5rG1kMcmIxxB+XAzGutNeLIVcQe5dH8vIKjwR2JCtswieYXMBvzsVWy3IvGtxtW0/+idkKIce1u+5E2ST17zTNpA/6vjvYJnmPOFAyHkVry+yxKC8KjfSxrB2xj8AWYZaxFkHwCAHD2B3QGiWj1bucVEP45+KyCEmV0JK4082pVR9F3YHnlyqcr0GwkgKyzqnr1+aEMUjd/9qH36TsXeFmoexz9tx2t/movCu/C+P/R3UAiak1MeD06/eH3xz9r3n7rdd7rs+5RzidnZKK87zgNSjD2YE5oG608PAvcrEBnMwP8DwTt+9/GXoDG7d3x2cffv+4PTtq2+AwafA34F7muByC8BhJdNK4QltTe66k9pKCxb8A83cXZKA9GIK53aSdP2uHeylC04wxfxFROL7EFDndGjUIYjPyCT0MTKfHxpksvo5+uWYO11nGmt320X8IhvKtxRbJgiq5tIQgqaCBqaRZzBPInaBScUbh4dd5lCvYV7blCDVIk+OpRiMKIM+AAqlqgI6ctpKtgi7c4xG+P81KR7gyuQkd/Cll+zRrBK2VrhXdlRsKI8He6CY57w1lThrgvFH87KsIlUczuYnmEObk+1hezPIBwIAKK3HOF+6dExDLOyF0133ZAnSnCJotOvAKluiO9Sp7/+SFCBgABGsRTD+TgAw5eXXwJorh3X66v4vmNsXDwyBdGj7jhHAzpiHMYERp5L4J/IiPvJSndzm+5MLcZ1m52Bxe/5sevRsbrwcT9z9dLgd4ByheHx/V1Whopc1AKw+M6oaWE35qFxFv1FI0E3v/IJHZjDl+SG+tYd27nNZVbUl/oQSKD3/y3Dy/PMpEUKb6FXla4K8fPPKQYL8XDd5iRRYiioD6xbMRscMfzA6LbZ3Y6jp2rqtZemP2BWmrnXUYf4w6lpHIz6Vul1vgaG1PkWg0xH/S7Rvh0ej/+9PcJC7+59RHoWjbar7n5ZZUnbJz0uWEfYraL28QaNSZ1Z8smxSQ6AfT1DlfoGMbLHwOnjQlt7zuyFGE8z54UzA2VAiNwOStDr0F1hZQ6Ztu5+g/e3hNjCPV4sUdr3TTyTzCJWDCg9bSeW2H2PTdgC0D/y0LtvGmO44ymS//nxcCJTO70AwC4BmPOEgI2KSVp79GO9UOMoNSmCHPX4qN+JPu851nRbMemGfETlM2eU77cYo7v9PyfCuYLIYjyxXMZ7HW7F7VVwneXP/M7yCf8D+4v7P0MLd4mPCFTArmqFjUeGUhjnk5ds7i5k5squMSvdFUwF2qyx2R6eh46dcagJwoyYUh+OpooqiAe4ziR4LTNj37FcBGFB+G9KivNWUQpNcC4OS817ElN9cYhr/+AB2Z632qDYikWQ6WGoQKML2CODnnE4v4FZEVFeAsROjDNFGQRfq/c9XIu9yJiqMCg88bmPOygTA25TW3ji7Ga6KOvTgdrNd+RU+mEYm85Xf6ccxL7YSosoOs5Ja6FF+27pmx3F7TU6stxfZOlIu37llEg7fkh61eGGbXFuYwk3r4VN3+wgVU0M1OodxxmdmzgqgrRDo4p46b6sSk3S1SryRpMBpBipRCtsfDy0eDOC24eNMcswWgeySgGnbpiM5D53t8Y7ZUnKoUBuWedj2dGJ1enx4dLQVAqUpAlcQzs7SjFQTnmnADXmM52bg//oGhOOidhDSiZPHxf2fSbeBCs9FXd3/XzxI4tBgQGzQikjFj2K1qEr4gK0OJ0Ns9ufQ4h4kxk5bc/0H0EZDU7VaBjTnQvnYABiHVQwGxSbLxQ2dMwW19AQGeHR4ePzki1//j8Nnvz784tfw69lkuUJnd7W9M1TVVbbAM8IyYqkMVbalHGUfW70BcfPsqipdYpCOGkQw/G4kJkY7Ncuo6wXofsGqRorLXuJJVbeluJkzqx3pTmvpWqeTl2AofY2Huby2e39Sl1Eir7xuYPoJ18DjziZCD7XcgLhAhXz6e18153Yrm5Q5no/s72LjqziDrXaGRzsHm1fjLlcHq7tCpZrmOlxhoje7fesAeFGWuTfm2u8t7/bgOrRZxNIESbRFccEVMSRCwRo9OuX2JYND0On0GA2OPzbZCdh2mLDgWAuVE6N7mKwqsDnxrKdwQDwb2dz/BMvESG4pRwSTDMMOuAxCi3DHBNvowa8sV213kLCqwiz6waIOqTosZL9RiR4d1I2zFWzsQPNEiMtdDFXH8hKWUa8OZLBiY782Lg1ecZlbLEahghVtCWYmohS0rngygaQdPmEEYIA5F1cxonNkNcZGzGFj5/vnvDR2kG5ZM62/xhR95ryCGSNb4WlWDmk47IsUMgA2BWkWqRPXTn1RCeHIbNXksGSJspHmpoI25Isx4pGrEDx0wuH5fwytP6U45BpeT4cmEbk/+eBYmU9W8dpTG5+4OpfsfOPg3WMs8AMiDSyWSr+J65r0G4Hx+xPXr40ryji2+oymMaV1iH5pMZCKP5Yy0ocgkcQ6cWg80q3AjQSsifHAMj3epiNNsHSbioRVBScdy8iI+C9TkgWo2kxZnh1FeV6VzbrjummtWBtrXbJa5qNaJ6ajexXc5OKaMXX3eWXLWEbaB1YQBAQqDbwY++4/u/uY4tgaMV3xofGHPAsgfw3TjQFjuL3GffgJvRlbIunmiv7SgK2Ux7RQqNG5XT1HNR1SwNOU6mQj1Qo6m3rcpxvbw1KKAalR24T2A4yW2Nv67nbePoBCYw9vL6dmCKQSLlmFXbIeQdwGI/mahME2bbNraQQu54bicYmxwwZbf/S5B3X00uWQgrSOZWKn2mijWmC73Q0UhUk1hTa1OQ7a47uNSiilidKezwXu06UqJ7qyT3oSQ3d84Z3XennXsPxg08lZ1T9oGXfAWwbz9k7sSpSQPyJHYjjRzcCgp54VzM6rB/Y91oVt8eOfk832nQLaHsWN17XH2+1bdwwjpAMtgheKgMTwTLaPug97OPReXrLFWb0NRNAHFowZ6i33mEFaDLIDC2P72I8iYjO+O2asjL7bjqqRAXTHu+GwUOh+/dueRxxU1Taf+BfzPgzrODLt+FnngbrFEVsvx8faqwIGovZO9UarVPAxk1X7SPAmhdYnop/IDUK0wxadMV31B2Gi1BVXHiKp4z+BGYJtQ+q7W9wn7gAK0WCIoS2EHA6jzaOnERA/mKJ+a15tRDGAoo6jGmXYloxTR78d6icGpo6uduHpwq0gdQWASvZ/VlDGYlq6Q6J/ZEuGP3FVhbNRrWGvsgoQOmI+kmA/dLp85pzCus6XobnSWZUS7+1C0z6/cUxeoFMuyCdIeXjqirPYkc3KiSmrhOIbvPkYm8OI87/qZ/yqM3m7PfqbdnoSoyWcABtCF7RRH06d7t/pHSvYChBNxRGovXE+BKbxyI6CU+8+Bkur3aFzRHtHBnp38zBHCo2Q7k1q2RxvkQFLm6JgtHtZlc4fG+Cp0t5Snzi5yMDejvV1MLmjtly9PDCyNLUJuultStgMpd/dF6KosuSiPRU+sPLb5q2h32/UM/Y/c36LEZFMsJ8XYy7s+cTMYNz3FLgE2JEi+EgykF0JZwV2GyYGay5xGE12drDIYY+PrpPZpg2ftQNl+01Z/thvh+faiG/HobqxVoJ511wn8LrPXgbCp8eI9ch2hYqHVvGtCV9PR6PXdztj4Mzvt3e7os56XGOmIwefR/deO6PHm36Az9xJM7ypoY3y/Wby+eF4Zxsd2BiSM2xBD25v6Ckc04SzxVvVMQILlMZw/TQrnzWfkcs4/OG1DVbjB63qdvm+Te+/9V7fyLp2KTFonQZkSTswgJUZREdCZx1LytxjczQ5DNxyAaosFQ6uXU7ZOJhvU6alTsopKcvDAVnAdIF/W3cyAdAnEvYYa9/lexRaA8Y/YSdznwPtEaOap4PO9NK6OELdppXeAF+ib04vwyXfg8j3st0k1m1vS/jFFn+F2dMu3ZYI6rseHDlRaDhENCzvf5aYi6RglIgUDOkmTSVLVVrjvhtmRSdlAB9JvOY4tmtPZbgCqc056FG8aFDf2wZ2hkgP1MlQvECsivmaIyGqg0pQhiOfjQDG64fk7KHX2RpWKNUlLEc8eCvOzPHo+58wII2kXjZFStGbmMPQ9uhRucR57iFBiGAw6ZXfpQwWGU8Jb33oI5ZZd1rOLDbchXqdDMDJXxUfVUnv/xXdpsB4a5Hc/7zEoWMoqhxy4ZDhZ3RdY0BXCQZ05R2vO9u5bC3AApSXGd01APo3Lvj+uzK/WQk6OKqT6LfDaG8dpIAi+UPo4sEdc1+B/quyGBY+oEty/5e8yWNDQLq4jlLhgBVlhkDR6gAtmaM9bYDSPYMYHCbC2BRSeDE6I8IXymHdoy6IcqsCORnnrwIXIVlnIlGQQsThle613aYNbub72K2iVMKXIlr3+KmrKLsKho7QYYVpR+qVnKcgHj8XmRZ0vD32/mdYpSwkS2BNEWLS6eSL58Hk6eFjdN4pFPn7k+Pnj8nB10Hd8AQLgQGeB/yAnamvUkBFK/ECGjfQibwagp2+ORbyolP6mFuZXJRSYMrmCVCHllxO5sSdTN/hSfTrhT26TGqnTXWXxhEDb9RjZ8ukcrYNXAr7g6Vpa+4LIw6MdlkW6pCpWQn1IjgacMV2v5k83w4ZMciXsSk49jkgLuFEIl2uY6TlBlhPF9IDFJpRhTxiomOkxs1/diWp6Q0DDgrPBRWX3aIukYDKekO5mW61qpe7DO7uyfODo0EHxtdtG/fWXuFh4OfktkB4Oza4mnF1cEcRBRbGyygronYbymTKeO3JAt4QFLAAVsoYFSkY7ofz4KhnLBZiA0v3g2akkjzUXoSfvuwcHu0eKBrJ58C2Xmcjgzan9xBq7NgmfRkeHfr+yKbEci9spoQFvHXKDH5XsiBPRuOdsTSbPu9tPvi8/UeR9xHzc8uRm5EpDD10gZrVrrl85nwjxNqJMTB/GfBoDsxFtbB7vShTGcD7JQU9yc9DLNNOg60FhZW2eNLDRs6e7Z6CpJs60xQZt8zFlnCGJrCtHmfX6tkYFbqL+VR/6iQO0JUAfOMwOUZD6rCLDUSC7G3MgYrjlqrxq3RnBHUJzBSnRZ9mIFtcVe8dRtx7G76+oI2nm57loOdfmZ5n06fzKePP1ZsR2Jf0zgykrSE42CATMQkEawIcK2ll4CoUKDaKC8p7/kODSUxo5+Nl0wMvJVoOJAwETskFwsPrhHPBKVAqEQAh0t4VtwZqVzfq9mypsM9DHeMQtC400f0vw89HDgzjj0KU9km1w8IQUVeXk31gxc0ZdNf9xNYE/dZXbCgm5+Yn2mDglUF/ONExY7Q1+E+bk0PbHkftnu14AasOguuHIVaoLQHop2X4gfuBKU47BTVj6YilwEsLrHQrZWoxZMtLdo3X9Pbtp6bIQAm0SZS3SiinfVeWxkQbwKRDPbJtsXS/us2mh8fpnTu+MCnN2etyQhcNS489+/jdD+QtRueSDwtYL0RfbmR7KwhZX0FLuaAz0qBDrMCi0XyYrkFDRehT8o2qu/joggPQCzxV8p+o2XaCBohe2HfXNcazAcZgx4PCsNWQpF4HnA2lGO/22vuCMCHgym/3C4Q3v5d/TIUBhpKVl03VoiBTQA98TZa+5hhPu4L2Mf6K4WkWGImFEsCGXpVhITrhuZlUg3LTX1iJF/s5cp2UFrYp4HVenocajS19fS1d5K/uVhiKMc6Trrc2nOK5CnYZmYyQVF8Ywz0EVh26UqZZ2LfKEG/5O5Ztj9xIUrM0freDdLvMiH0AK+pz82PdaE6l4bT6VUYm08rmYb/PxDb/DZxtdOn2vkueNX88jcbyzFkIw2HU5e5cIEsWH5YOZDsBlR7r4O4B3Rkhf1iP1vGZG9oXZp2NHaukT1F+5j03NUqsdz1NvDQ7FJzGFGWYdv0lX39urlm45Zu9zPOdOwhLYtZs6KL94X7EfNH5RSK5AIB47h0buOYzX8o+BKBUhpsVa0wRFyhwfMWD2xbsaq9vo1dw1IFOtxMB4VerTCbdgJFNIq1P3L9393Ha+4CffUWcfTf4oXD5t7471oFq+64TOlANcQ56ct9VyxBoPRhxX3hn5q49Oyhgroq17ssdpk7gBYI6yVnPlC+dDRRe2hynlqIn+F0SZPdiToynspKs7QkfGg1M4AMKKDVpo04HjGgg++pT0uTIIppHNC3NXRp2FZs3uCI2YkRY1XqcwDX/UIJOj3lknFQ1PjaTj7LppbPj1HT2VO8SWXylb/Lr3iGLb1SGlXWDrOQXnL1lGiywE9r6IyP8UNz1RAnEX1uCM/fD945z4ODtJl+9efPNNjy+e/vuw24Evnh7+vbs3Uex949vvnuHt5R+eMXdtvw3xOFjYNPTY6r3qRx5+h018/dj6wZz+3pBfYmgo24RHOsesHP2gYepONscQmD09imrMn8x/eHt952Gyp/Aem/YlfvyjUEHp9u7370zJXxSyj3jEspXG1tMtJECS0jUevlhhcXjk931BJjC6A9b84yvUtsBAycNACuw8MrvGiYmTMDXgx0Fz/vrBVoE9TCOrNcfpUjVLmJCp/klZiqTccgBCVjl3d65L2MzEfihdbBsL9J78h5/R7cE6y561058RwYu/uDZElhTl+5nztfdAAusfPf/VtGuFaMyXKY6GGZcuu9N3EZ9PRO0WuOGiHc/E+cFn5AvpCOxFn6jHGWeOILiwtIcqJcnwPYj6Qp5CZRo1iV8xK/ZEXjaQWCkFE8FxBN3aKcbh/GsHs+ZR7ePujlpB1GGVj5hrVVFbgD4+8y5ZRfSoyx9NL8zT6Ql8PIUqPSOLzS7/+lK5FNTg5ywUGMydvit5SjigvEzVR3++pXtCd7uQN4yj8ePwZxSU/jPP/1vB5+6/hoc6ePHIOHWhmKblY1nFTFCYPIVSJP0U6zABvh//+GcIi1LXdtkgXG8e9tSyj8Uopc4Dvc7BmFdYTzr5OD5Cll2mV4J+mE82Gtx/FHdsm97xTE62BnNp8rlKt2u2ogu9pGAZoXf84HXM3Hez9TVcfeSMperdSk71zLhtdp0xlz5L6e53nxpv4Q1eNd8i1La3lRiGqgSH790jlwA/D2KbQW1G+OuwKSX2XnB33bY1hluxsyRzy0t2i2Y23M92APgV13AKtchQmGOYHPcxHk7rY8lO3amPAZXp15NR8/4kIfT1b3B2OkyZyRwxNFxvTkTqTvlfY/dBQYNKL2LyGfRk76qbQ37cv2tZ+45hhDxa73oysKHXb/Y7cu4S9SZKQY9dU+/ev3Bef3VO+fzw+On0+PD4+cnzgcAKZ3XYuO8K0H5OkfHJ+R/LJ0ljDm5/ID_RESTRITO_DA_EQUIPE/U+4LXXifBkv7v9SozNMOKt4jQ5R6Rx9oRcJabkxOss4ZrVFFKGq+QDnR7aaLC4P3GQytykJU3/NS+4ANkwqP20QGVX1+bvSWoXY+fow8o616KbhbtTXShnVxS3WGS5pepP7kXTTIuzfm27yFv0T5Ijh+06uZDfWbEdypjwOvaFburfc4g5WreJO+6P/HVjC005j5xbXTYTvUxTrzlfWu/7KLaORGTZfs/NC35uDmQQtVFohOvmLcSOVVqZ3yzJDdzV+2QzeXQAAfhRVOVFfOWZTjW6Q1t/Whd+Cht8fFtmmIX2Brfo6NGppD5sKNBT1RVlN4Y1ePc21Qr602qqCv/wTfjvRh0CtgvbAkylUYXPzrNzAHYzyq729DO/F5usyYUMRRfh9yFGkjl+recXV+TquJE+NPlWhLpucVufkD3hL5dApV5jEKSgX9c5zDw5K/H5i/AqzOOGvv6KrWqO6wo3eCZ6HC1VL+oNtpRkyOVMRdXQldxQtsxwGjDttvIACv46Pr3xmlFHSCjSfqD4ZCn2pWvc8G/kxHybw/x9QSwMEFAAAAAgAzbtQXeGFNJnTDAAAkiIAABIAAABmb3JtYXRhY2FvX2FibnQucHmdWuty27gV/u+n4PgPyDHNjZxuOpXKnVFkOVGb2h5b2e1Wq3IgEbIYUwQDUr4k3Zm+Qx+hPzqzM32KvEmfpOccACR0cZLdZCKRwDkH544PUA4PD/ulEpUoav7pP5/+LT0+5+mnX1bZnHvCU2Ih1KdfinnGK28uV56EoflaegvFi/mnXyqvknk2z2qeSq8UOfC8X2eliA4PDw8WChhKXi/zbOZlq1Kq2ruE1wPzrIR9WhfZXKYi5TXXXKmcP1iWUzlfr0C/diaqllyJ1BIMVqF3WYfe1auXA5lL5RCKYr2KavFQW9ofTpP+m9Gr8+Syf9V/ddW/fL1NXdWPuXDIr8c/vhkm4x8vhw6lfFjlluYCnoe52FIRKaKiskTvi4ODs4vzsRd7bJytROWdi3vvSq54wQ6uR38bwkTn5GA8/Os4+WF0On6dDP6CQy+iZwcHB6lYeJWok4Usal/otYLugQd/wNGXWeHxPPequcrK2lvwVZZnsAIvUvDxSt6BOcVSqKwGn9VL4PZQUEVBQhmlgqWM2OgG1pEq4WmaqEvlB0RB9EBUqmiRFan/vvDZfVed4TALNE22MGQZGCcLofVzuR1Puew9FJsVlVC1/yzU1HZZ5RV8hfp7E8areZaNUX8WemzZL6r2TfCq7lcZbwbmFVnKpltqRLyuVTaLSlkaK9gRLhGEpPST6zZLuqvphXbXgFhtC8fg7xNefUAh1YfBhhiZpzveNqKChggcjnTW3YZpy8mGi3zMy1KAOKAL3JUcde94zkBZdvI7Fpi8m8tikd2slUhSU4g+PJjsQ1sqMUdTYBAEzetMFlVrCIxEJb8RyX2W1st4sPJPOqBMM7wU2c2ypvE/RL8PNvhqWSYrrm6yIsbXXCxq+w70zzeJZ7Ku5cqlVyjZYTjZZFgKngqVpFlVQysTmkJrtpCy3prqBE6IlVeixS0p2KL4jeLlsup6ZTTPBbeVg3/0UnG7qkM/eTbtGYIn+CKeZzcFOj7e7V7R1ejV6/E2RyMeOoZa8RrSSFV1kmeFSCChUBTY9GxnpR0+4qhKPs+Km7jTuiATeRpvZdoiT6+h2+UCKpoInLyC6q4VZBa77L8asnZdtS62xQBZr+l1MN8SYx/fpq6BGsepycesw3rAYhMdx6wuZggF9oy1SWlHiaStTrMFmJzGFyejoeqWvMIu4tNUyFBTFrQUlGE4FeFMhPUXY/n3nMEq+yDiy9rH1h+4E3PcwSJ1M4vtdgY9Ef4Gm+KtgzSn3RL29JdzDGTOQvZSpo/eGDzi9pkybk2cINN0Jwda2s8n4p/eXo9HZz/2yq2kib51JOzNw0508m2AfFjgbnrCmC7ivTlbRriGSGYCFCVnEoce5Asoy2YMmo+8T6CT1Urm8VitxR5PjbMacjdkryE7QHOv4zyfOM/P4XnAS2x0ZlQoeDijTvDbnftEif4Wv2wGoLfrqM7JHk91TlwZt0KU0LTrZVJgZZGbCllrV+0Y/bSPvyJzBsPz8fAKC4tKJTaRAERSiT2ALXozPNMdz3Gv4ZnqIprBrvZUmH9NeKfdnQjukQ+Ks6sWKHv9l+djZp21r4W0YxHCLHr0d0WEG+gzahygw0RcbobtCpiaxjLjlUh26U1rmLbivqJhYep8rvuUsR7czvTeF5KgCeqe9N0qANsv9lbMMc0efH2D0LVA6V5LAL8AlCm0X2gbcFzB8OaAEXwnnhZFA9ZWZu8rX6awpwVBF3kQXoNrcPcKIg3OfRzXGitRrxXljAFfAP/Aax/2Yq9DPLqV+SNCeuB8vxYVAnyC/48E/jFmnrgThVdLb/lYCgXOutUHA9ry2iOAgkKu4glaYkyYfhbgEX0EnQG3z4mDbBLDHjroyI5N2w0K+VEoyXHhs0JMYGdcN6IPu7vAYMdlFb/bdFeIR1CLV/f6s4cUMR5NfaKl90iHKVrdppny9UtFSRCKB4h6Im/pLeiRY2BVzexqhQNGrZmUt4BFb/2mMEKssxCOnDOhjHqAN9UOxLGc1ziJ4Ai/XWyVArACeOUbUbsUuBDQ0EGA1oGgPbXKsABxPSD43Aq6vKwhCKSa4xutjR50Jg3Kgn/2UJEVEFgIBVav6xJEbCFg7qW0LkGCbV2bVAZN8ctRVfOCukaIjsbXokzq69uOAUrElNvnYmvVrDlOYR3vw6dU94RP8cMFqFT52gQXn+5zHhJZ70G7y+qM55V/k0F1G0/V8lYUVawEHRp5nvuKTf7+0w8/pcn0iIVEGi5yfkM0b89Hg4vTodZcrut40hY8ScIa1CI3sC8NRbm8h8IM9I6aImpKOX5I+qjonT4FQs87jkjpjjZU7KZZ4YADWNv2EWqmtEDQLpVVa/AALoZtKxeFofhj/FwjhAm9wzlqulF7zGPRO5kV/kNkBByxiJF9D6g3rGv9ydc15EqC9VH598adit/H/j2G3WeaoGKBB9yMYecuc9hFfOb5p2IuYH9NAzDRnGxIUDzhsCmorASiqAesuDKnziZgD8ozyDvW+8dP1RHYBZ8shBXJ7k0+HZa5VKVUvBYxLx79isTw+yYO1KZ1NHI+Q0KJ8Iy2rmy2rlH5kN2LGfQ/nCgkx3iVdyl/D99SpBmbknUoVS8fABLrv3zJLMZpVOgaD09aWq2qdfTUxAE2ZSVSN7MsDCMPbeQVaGEntk5SsBQ2SspfwqHGe8AROsdyzIVNzlkGDdthgIrXsmKcmRx3pj0ttEkVGu/CxOZpC9TDvMPZ4LsTykMt4WS6rxJ2cn5DmGPT1sLHJ91p8IRKJ45K1rG2NRhpTZr7eKN0tN0k0Ap60lXD7LWdCaYVijkwYf2344urUd8bncOhbvx2MLo477+Bt9PRoH/a9877dJ81BOzoVlCV8Bm0PVtBugy2iku3KmyGPg1B3R4B1K89nkcssK6mqeA7U+GsZ72hx21Gmup0szwBpAZ7HaQupDSlfRcXOwKH+uCxgLk244w1YD4XVZWkkN6N/vgCZYwb310w6XaeTSmHb8M7jPY9gpNV5ZPOsAXhvlfhgQkUympOMInkJUy3rjsLtPmqjFf8wSf5xE5PxtaTZycvjjvPjjsvtK4rsG4JarB3vIggpRbiDr9gr8YvPlP6LcPW+25NJO/WOc3dSPyCbQ6/sM9S5RN7Kj5EBvnX6rHN0MdwFaagXelniOFQV1tux5AxPeO5BfuY/ux91LpNVlAx8Pb4s9ZYPMxFWXv+9zxfi6FSUoUjSK4Hegxs92CdF9T7PTSYmTDQ7w2iAHxeojubUCDWhRMNJCckac9bF7eFvAd8tZ7l2Zx8jT7keN3NITbioaRfJUKvAPirABPPFNJBBjR33ijftibKgFRfGIUEAs6gcEQIcQQ5+iXY3AaBsksybBV+ZDjGupgudPEUMpTEuvgZMi2KdfX3z8EXKgS1sfXxdUXRVIVL2kU5TXlZewi2/roi0nJ0CbUaQpdCEeQH7KcT7YNpBA4xtYBFrfWNPMNZ41E9Rj/dA4M+twd204mU2VEiQ30LYEYTa0Utck9qQIHtjmz6MLE8Cq5cFnw3lJMqSqOpEQ3NcC1cQhrQlGaEMqyCM6Gz9zdbnhFDUCB2GXRKJjRhF76OvBxW1m6HTqnqbA5uYOaBdj+0lXoFK5WcC4EXEs1NAk7aoLUC2qykBEZvHpGre/hOBoXNdYUepO2BdNv4LcH6QOZrPC50DekdCDsywTJz02AfY1ZVa4evcPj01H42/Bmg2r4+pUGbIppi6qCu//3zX9DDjtnmNt3YBkIjnZlAg64jCSYN8WgE7jmiseBghxmT5ajJPpGjFDcaJhKf9zsbFd095dbWzWg4HHqXry/GF99fvBn3RwPv+nI4GPXfjK7H197g4vxseDU8HwxD73knatUKPZNHTTFZkRRpyrTLRlnAfG3oNxWIrKCujdL+fEc9wVl7PLM/itq60on9Z6LnutkFb9jDw0qUYbWexeRaPIzDAxSVz7pblhP1Hjsz/M201BqRleuZbTI7zti0yfEHNkC8CbTV396Boks6AO5QgjHVEk/RMg96pSu38Wz8tLvbILKmIW6oSh480vY0vNSCmxeL7vbFrI1XKrMmWqcXo663rOuy6n7zDUxEUt1808QPKadOqNYqj332BfJgeymtlhkBERtNVvKEhqyO8GJVy6pSFp/+eydyT6xsrsI8ebiPmE2aiU0E1+rbXMXAHrWDMnC7QpBhMYnZUSu7kbUHq11k0pwe8UbAztKFk0Ut+gcAnG9O9PrKNN69q22vxlD+/hXbClFxSXLVusDbk9rqCwBNX07rQQIgUxgznUiPGigy3XCPvT2Tqk5uxWN7DNbzzv/QiAq6N84+QCmfn/35lIVb6B/wPDtyt3YwAnh9+0M6y25ABGwVUSpoIojmcIJegLKQ7v8HUEsDBBQAAAAIAM27UF1Ur5saDgoAAAgYAAAYAAAAd29yZF9yZWZlcmVuY2lhc19hYm50LnB5jVhBb9vIFb77VwxyIYUQrO3sBlkFLKAocjaAE7mOkx4MQRiRI3liksPMDC0nwQIFeihQoLf+gAY9FLtAT0Evveqf9Jf0ezOUSEl2dgXEITkzb9578973vjcPHjx4UUvOMsFqo1ilNLOrr5VMFRMs5Xb1JVcLxXg+57PVz5a+Y6oWc6FXv5Sp5IalqqhyYbmJHzx4cDDXqsCnPBeplao0TBYQarFqzuvcZjK1fk6m0tv14HOV1oUoOyOxueJaZOsJwyI6646Ksi5iK27tesIfn08Hpy9fvJ6eDc4HL84HZz/uzeazXHSmXwyenY78olej1xcRPg1Hp6fTd6Pzi5fDwWk71JGkbot8LWOM51EudvSmGXFr9ofyoHmaK11wy1OupnxWWgbP0f8HB2/Gb8+HoyQ4UaUVfSZyPlOar/61+id8zZn4UMtKMM4qrq3U5P4C3s54pgybyVku1UKvvsxxNAYqsKFWxuCAcH54G1eiHOTiloXHh8ePe3FwcHCAo3CHxlMbQueI/BjNVJ4lJzw3IroWomoeUxgntH/p9Q8YflWCNTHPsikU4gvNq6uw97SKN29Tb2k8l9rYaS5LMZVlBkHJsAgPe17I/nQ30UApWS6So7sE0qCY8jlpdGbDo+N7ZZEF06W0V9MStiX02kzluVyUdGTJfsTEQxz26JxJuMfZjbMw4o7Qik9HJxdrgXCErsuQnNiLnRfpjxvVwta6ZFXj87nmhXAeT3lFyRFZhUwzkan1jfjY+Ld7Mut5F7oW7R9vtovnzVm4t1CrpUmOIqSfSR51pm2bvRP5jdlPm5m1VXNp/ZE7CUuZ2SuTXD6JD6NH8XH0Xfxk4gYQppnQJukkQhgs+3aWP/MjgVcBh8JEtkDylewygM1BFORibvHfTFmrCjxoubiiD7I0MhM/bp7eBRPvFvrd7m4VPCS5vaeAC2HDDyV9u+F50IsCgzDKRbA9Zj7R0OOdr3CX0jRw6H4YbSyLeYX0ycLbriunsC/GvzO9Hm5mt8ZCYOS8RhZ/klXoV+JzXZTGD5ke7Ru7Z0oM9+BFXAkOeYlfREd6eThpZYs8j6y0uVgL99NjGjDRZfCmnhnh0QOoYUxdWgV3ngtTAY5XX25EjtfxTAOAkJcyBZSYYAJ1IMBhauLENxFcCW7vOOEf3aZwVbP71Op4IexU6akLR32mw97aQV5K6x8X9qS+j//2gJ0NjeEusdQSUtzX7TnwSKMqSbgMZBZMHgYseNi8OwuCyVM/+Wh7slqWUH2ykVglftrxpEWSjc9dBih9jQRYOuWXpLjPWMQQlTkqU3O2dKEt02CSdLVqpdBiGbnlVJCE5laETnSv9QD9IEz2W2QJItYk0vpHdSOWhFAlzwlhr8Mqwv6uLIsSIJmqDObf9a09BRnBvdvatCHXUQmv3z7enehIeWnfVLm0Qa+3ZbwL3a28INE+btc5sWWni8idDHFHGt8I1MIUxm8B2z31uwG4LdEF1wtZ7gV2+orrHWev0SsCsNQew0IPYk8Oe1G4gTD/5nHNPzeYhpfJtl2/GcuWQCVjdei27m2P2Y+VINDKbjnS0NuzA1hbfpzadOsI0+4R+uX7hldkr1veZsa+LfNkvwY/nd9HADCyXev35TWlfiYgS1Ctd6t26v8hsY5v1vNNme7+KLuS5HDfirXNyDmyuorxYPr446s6Vd69Au25W69b7V0ONRV/qSWyaqu8Y1Xi8jdV5Vwuai2mWcN+wzUNDpvE2SdawfnoZHS++uvr4cvBG0aWsgE7efv6+YACfbD6y+rPY/Z8zIaDs9XfLt6ejtnJ+GL8bnx6MXg5HAP3Lxww3it+ZKxgC2oINHhoCu7Biaiq7WpBVNR9/1BzqxVz9eY/wriCw45+ADzONt9AQkFjVv+2da7Ym7MXo/PDw+8eE7N9e3L2LGZjJyYDa2WFQrDzgql13YrwqDuFCxtgWrr6mskFsd21ajG8kEtjeadRwXOFtXAnZ6R1t2nZ9CysLjhb/bek6TfiUwRTYD0mk+iuFaKAloLIOXokUHADdlhbmctPEB3f79BxR91nh0eg8c8ewT9UnQm3yakE46rxaa4AapCpyZHdLsv5aSPpPCKflXDtxkbqBnBYzgsR7NPebxkUX/3DILCxj6HOQWUqX31dwGJsTlkp582pkhZYhS7NSYP0Sqv3WEHubdqQTNygL0ToCo1eRDGqq8oZAYNzHwDV6gvAhB4IpUvhdubur+sdYYN0BqbScu/fb3mQ5eIGqzllBsUaSi7iUtUMjvSB2e2Nnhz/gC0WcANFVCayGqUoJYNgBRqm2pDh4laktXVWN6dIcddpndZ9U8yG8BxfiNIp32oMHyO5Vz9rqSjuO1u6lrh0bnQuKHgpUhwDuXfTURs2fPOOwo0sdd4Wm9TDPrmQAJNNYJUw3iSfj/rBSV1m3hN+17TWaNuNylG3ouN+MChmkhzOqFIhFHLfSAbRo37wCoGbw5SCkg8qoogaN/ZdP3iOKCpg8xWxRpQb5IBf+NOGLzhU1LxciPAo+r5TqoGo5e+P+u3ZLQDdWvDrsFP/N73PPPiDS3f2ufyJ/e9Pf2fn3csEaLZmrzT+2Vl+WU5+CqIdwL60nkt26Jink6SP9eQrhge0NdQJhlRHy97DAD6drDuunZjb0fuuTncLgIGnxINRxNhR8KvVqMNCuiUkON0DruwOyKXoaXByGwvaHC5aqHZgso7Cp2wb/LYuDQB/PyOPsQBZq/lcOZyDgjgF5BUBHrWEWm6CsTZoO9vbnBCUpCWVd7HjPlZcLhs6TP4MW67ciCwldk0+N5P69zPtTjjWxUzoXT5tlMas0AuMHW8yYS+6Fh991aXxKd560VEngokaJ3eQ5c2EaxxEgpO60SpoyP66ak8dEQPpD3I/7C4N0G/lyPWvgAcGlEhRYfxK7EHXQYGzDSG5L6nXSLjQfMZzSsiCAXWpjXMCAMqpEBR1v0HIALi4aBYSRqao/r++ak1EOrp3u6Vu9HbyAe0XuQ19GKV18JCctntr4STcd7XT8r2j4zuvlHYY4ONWpjvcmVLXYLLUDZEikQ+RzsY+BlxjtT5pZ8Xyzt12bpE2BPAOLwSdS7P7ZVkFAn4F3bdFtZ3ewNUqD39lhzf10QHusNCdlV2uEjykljF+r2S5yYdOBrpkyHkxA9be9C3qowD/r0IU8egGLkbnFhJK0s/B5Wa/XCx4+jG5NC4BCZ0oEQyCcSozE0woykwXc4NzgG23TngJ3eaWDT2Q7bCP/pYVfllHGwJmcBwxRTI4lgEKAZi6UjoJRp6ptFSTeGZw7zJ3VZAEW1XI81s2Xxfb9W1sl8rOlVU3CtpusuMu6TiX9wCuXfnkvvaeBqHUEjn/afDs9UVT4BUVzzJd/WKozkuiIFmzY9NyeGTjN51WgqKygU5V29/57YXbfYrdp+6WKOUqpktrOPX/UEsDBBQAAAAIAM27UF1DTXnlkhoAAJtZAAAKAAAAdGVtYXMuanNvbsVcTW8cx5m+51cUDAeQoCHng5IsL8EDQ9EUs5LMSAoNI/Q2ijM1w1K6u8Zd3Yw+TCC7pz3vHoK9RdnDQgEMBKvkkmPmn/iX7PO+Vd1dPdM9HCsI92CLU11dH2+99dTzflT/6idCvMV/QnyiJ5/8k/hkuD38pOcKcp3HisqOlElUnmkprIllJpTQWSYnevGHdIxC9SrPZK6yTNk8U+Xb5jepyujt5zIpVFwW63Re5JbKH0t0UExUT8Qmnfk/ZewK0ceFyRbvMm3ExFihU3RwKWNjt8uWMKQLw2M+kPG4oIFJMTdWL/5n8d/GD3VsEvH8ZB/NyXWDdvXmKh2rbFt8KeQbnRS5Eov3eJLmcmKobZnlOsNwRGoyPEzRiUpzjYJyrNXg1LcFpmdSGt6nn56djY09O8svVC6jN3tnZ1anZ2eXMptfaP9jouJc3vEVyyf8g5/4v7EOM9k7O/sWzU/EUfR2cHG1R/+kV2dniXx1q9lTb3B7+9NPqzFd6gnNkEW2eEcyM2K6+N7qsWEBjI3JUENOpMUsx0WiaOrSlg3EOtF5Na2nJOWLxTthzi2tjRO7TicKckQzkBDWEeKeUT+8HLuQ4qxSprnBc7wqxm79JrISX6ampCO/4l/4/Wyw4x/xj7vhj3uf8N/f+FenelZgUcO3jwbDKFPjIrMmqrqvm/gCj+exTE0keay22aAp8lJlVzcCrUNmaKLQk0uVQgh/UbWOfluo7DW96eqTdpL4oOYzk+n8IvFKKWlZqrnH6pIKIqhn0phIs5V6Bq7cT+11Xf5Gpeik/t1Qei3j5kTnRYamWT2+KNKJ5OV3q5qrxYdMjyVt/XSCP1wxph0r7NcMCjM3mcgXH+ZQJmr2qreKLKNVZMkkVOUNXicpmGTxPqdurOAdOTcpqRGUUfJz1+vGAHPy5T6gBcolEjXRE7nbaBMKYVKIoVh8T/qOhUx5l2ssY2ZpdhCSxt8CGn8pW2DnOMQTIMXc5Is/8g+SBfBAoUu/vk5eqGQLyWA2yzCulCSLIW2Lo0fH9BQ4ot8Q4MQ98fApFwFrElSVhmSk8X9ebEUieuhemuhpYWXw7ragqVuFOU4VwE0awfpN0CUxnvRlF1BhGHvotwNL7qDDCnweEYwVSaQBQ4Cqh4RSIo90B+A8wRI4IdDYFAs3YckCiuexgV6SiklBixGsUzv2PFcJiayPsfZJCrTM2rp1NnY3WFzuMuXl/LZAbzGdBCFircGcYQAzwxBzhg+CH6PPrwWgLwajKJhTBBHUDRyNdhoPnWQIdrtg6BcedCCpWbWBmjumaqQLimrcqd/DRtOQXE4aBZVSYhabcxlvAEwrKIYy30hYQI3Xv13rgd7eMBztrMDREzPBOeQak3li7HTx3veDI1nLcx0DR4id2Dmq+d2b1SK6FpOeU4N8MEIHejWtoRIPC657LAsOCKqNLc/Hy7nKaGotQPS0wMljQZ5UZqxdfNCWB0wimPzw2/9avJsBdBKDBcFcEmGEPwwdlGwLYIUb1hzrqvnctuNMMfYtTRxjZNB7R1sVckiJPGHlLR3tFgriV4IGjVKh48LSmaMvUUIoJUEOUO5AYHVSTTg6jd6WyHJ0BbyZZnKMEoNxytxkKXTiLc4M9HR16zvA0BaokN4aXn13+6qrGmrdvurAqIc0bY9SHiLGGAt0dLKkAGsYEbaNxCF2XgCixhJTtg1VgkhI6rR3U6oREwqa2MwgT5kWhPSiSARaSJiZbghPo01Y0ChqTKLkRJ04c1BqYMd6qubpBn6sQhLXjTgskPmFgkBqueavNwCa1dooDJoLCmkzmLguGMemmNwwxNxdhZjqHGycgZOlTbkxoOxndKxhMU5OH+7/oocjkCg71qsH7UuDQ1EQHUDbhaOoNJN7/eE9kei0yFvNquPUzhVvgMxjE6tNTxQp/4EuYjkusJ17wsJewjZNlcdHwTsD3Xj7wm4LP3Pfty2wO/FwrNGgxeH/Pqb6PQxSpjBOtGWKI/MYkgavSvBgjgnQCkFibgB0UIHg0Im3KzDFeQYAz2muCZ2M1A1Jmkw4UF9sLv3GicaiQgFZpxJSmiiibJkGdZuDPLVD0cHecDA4O+s9JUjCol1kyVtSeHt11Q/LcDIAddDJVRfK7BcTDVQiqkPQaXH26vpgqQ2vBtltAs3D5bdov4AkM8+BCZsz0pN1ag2GCISPSWWZ/wF1Fn8mO4E0EHZIkdpaafgEQJUx5AN5gBjEKlSO9SA0GIWI1ICnxpOdJnAFP+5fT6Uew1qbxsUrE2G1X2J0AZMafh4R79M2V8TDI1L9sVHdTOo5trIMjiynHWgYB6r2ZL7tmFqCNgY2NuwSJS2GTAuIbUJLIbF1NkG24M16RqtNEGq9BukjU09ldaEfCHZ4kUHPbhLmRi0uowOTVrudzpCEjr4VUDuVaaqspd5fXACrVuHtBQgWneKl9pLNjbXJjaf8u8wknH3XAmHh6zI51wQVPd+AwiFr6EAmdgUKRs4iEgEBXkJGAr1Z6oZIFh/ADWGp7RPqWD7hywl6rjBhFk4bMPWOK4WRwX7jDnVJLP0RmrBHCZOfYa/R1PzwxOL3xNVUzA9K2ZEJbD252tkedYDUiwCIzPzqau9FRD4lpk1HIUhNr66u3v4yGtz5ZTS87AKqALD94CQzyzwQamlntOPUL3/4t9/iBfzzr2IGqRADIvPshROuJ1cwYSW23Cs0VGnfGOcBqLhDahBkLK9X2FIgoaNpDT59tgEvIvsrdUgR1TMNcGVwNwomXVlonbByaCEDHHw5DbRsMRaKEbmG7YlqaMYquswvTG4uDQ5WEKV6BAD8KQAerY43QZbgxXpOsOrJrxD4huombxY8Vr1C+/G5mvDxVO4JD7CXhvdBTR0+AlF849h5RaK5l1a7mTqqXEEtyPIlzk43yldES9AjdA+HpXcH0YOUbA8gxlOqpLIxjZilg90PalV6qnuOAm/BcBj0hoP+oDei/90dCJzEOMh4sOtcV6l0jh5/XtH77MN2nh4iVLzHEpOaKfacjHfFuXZ/efrwSs8IA7EjoTzgSH55sYBKE3J6VOziRiG04BwyV2SrZRcmmoEwHT06xv89CA23vFPpHIzj6u2oC3oOAght7Jl2nDl0a8Doak2escKxw94vyyuP4jQ6LEjpufZL6BgyWbBeb/lQqRiho1UQujpXsM0IiGe0E5hXlucA1yAF0Gvd2Pe7XEobeJGORqPIjTj0Xa96ljrJzjL4lBoceLEv9HzxAQ21+bAbaOTfnWUGonBTH+cburGXJ7H6NnmHGi2rm7XcRi3OoecmOcdeSpg82OJlSdhx+jtHIx2UjpV8BCyVGs90RGLbl7y02nz0YPDTnhj9FK3c+ynUNSZuaIOzVLeZcfs0WN7HHHBxbZNnJy7eEMnXLlajnDGlM+/U8YTH+lMLEoNuAVdsxW+ss65qlekeOjiNF/62aMgRW5RwyA1NpdAu8jol2gJOxhc8KDNxQjl/TXSpB2WlXQjFIFMvI2N0arJElbuUbcA3GyAVkaC9W8MtezssxSQ7LbfmBsK8DdhI7lRtWmqhWeOyvgBxZOV06qhxLr+hlxM5Z2vQkmjIl0YTsUD3wvuueY2cUSfJRqOAiHIqUWmiKfz73fAzavitN3AZjQZR2X7kqcpYdjuMmgJSlYYSiZMhVLJjLBYSK5gs3pEIr4Eba9BqOhP2AkqFf6GYtg4OrA2Z8YuBoeRaCNzSoOOBcUVBXgyNqt20GbXqLaoDjux7Ba2MK/8wiEcV7K4DO2JaS201VHY9Dh2nY8irHv/o3t/+RCvpA+IoGT4YoKiODxOhB5u4YNYUbIp2LGpMYqKmOm3ORDlIclyGXNM6HBDRGLb1yMODVS+Lf9+Ifh2yd94jjEOzqY4JAu2S3CqKA/4w1hZGGSZYR6grANul/gA64HHJ3Hnr3V4DB6DQmsx6zAyk87NcoA1u1Lz0Nls9PHHRx0l3y9dks/32JmkDeq8ZjwuI1B2fQ1A94V/0hCvdOjubySSRkd0q/5K3KY5HYbyzMxXHexd9vC3xkoznF6jYAYCBOtbpASxSoD4fBlgbt5/bQXC/pMteuwIJYWkDAbt4JeHGRCdl29UyMvjteneXSx4Qxq1IebCQHw6Me8pYK0UBkWycXvD5j43uLecPBPTsbp1xYKOfHZ+cdnvbgwQC1sEgGO68qewwlP44VIGSXwOeZg6jhoIKOs4FFNH5UCkNgXf1xQYwSq/W0wraCBzujcYYVHjncq2bBNKdFn/Ui4bJ5SyD1aj4Cl4eyXNMNabhHOkirrJGQuLWaZj1VrKRAmxpBUfviy4sxYGclQhtaES6eeTMxNCxy1cIAPUljBoApI8uGudIMoEpA1sB+42OEzoFGnFwxtoy98HMPXOjmN1DyADHrRgNRnd74jA7tz/89j/LQvSlMvUmKOEh1hYva6vXGO6p7/j/BhSNyVhHQoKmhISw8njxvri6urNqjF6bkUCB6dypXJX7EeYktGPZI8gaC9w0qYIAC6PT4q8pbWFidFOFM9KBk1sggjxoDWWSFYnzja36u5r5VE1HwMfYmcPQ3z5sVHuwAR/8sakKD7WcpYsP5AkzQUZfI81mKdKtXo3jouF16Q4kVp4UhjMCKTXhXb2JGdp4OfCMcSsBI6TmhJkySIRueFL7Gwa2VV/Zk8pVDKGGntnAw/oxsMYAki9732vf+dgo5y8k3PtCQg3SVkhrcxYzJ5OU/eTxScXurAMIkct4b3RPfNW/lfztf8U/3/b+4737vQd3xVfCUvkHKrceyBd/4GiWDTzMJNbAJeZyE9Z4XhnlhvfQ1Q4FIIk6UFQ2afiWN3C1L5uX/Vulj70rD/OJE4D3aneE+/axHZxvb9L0vk8qM73vHeZsdJXPLr2rolyoxIFITd3+Tr/5j/SKv2hVBch7RvEUF4EL1eoaVpNQso5yep6TTRg6yFmxNom6uUZaXeStLdePp6z0Kx513/XN4kJrKpPHhXJzySCMtd5K3AQegn13cHjQW0quniqX2de6/Vu96R4JJt7j4w5OGmdG0WgxLsBTrDgGzTjtiRP6h2NtOIrceZos3lEAqU7CZIJEEfrG0CygAOjytz8dNAdH5cPtwWAA4AHubItnTitpODioZq5NFyuMJQ79abVUqgFCztyESMitU4bVOzDkeO84erx1HA1+BSvs1fzWrdM7x88ie7svb28Nv9mqfj6L3tqLq9JiO9k7PV6PJhWcikpwVYr1Gu/UJQ44NWFhHDsBv4J4dOIDf5Mm19zlqIRrPfVZFpaS95cITBA77Uab0C/VML+Ggw1w6F7kxhFVzCBM7z4a3C+fBwvebYYtC4x2ZuV+q3bnk5OT61xWQI7YqbPHI2bGHLffyHMVvB+4r7h0a6k06CSI6PmE1zQXY52NC53fMCq1ZD/VbKXdY0V+lrHJOKeH4yuL79NCfgxCYYEC3euJ0faDB4PKq43fd8UsKyCAvqOixuVUs3PViocHZYSw1Y91mDh3j7VmXKa/QUkobZvSlchhQSaANYlMaEo+v1K5SJ6bHmcpkUbEioJSX1a+j1E9TLH4nbgvKNsgnTmR85hdG5QDRB53MRyFE0uCDTnVOJIoJUGel44R9hubgvz/elY66tET5PWiC6hOI7n3NLKnUVKC0DGXzI/rkhNXB4UnYWFNiB4egCWh1tkZueijl7coGKjiOHp5DTmSM1AEyYGqWqVV5WrqIE3PndD6PC+atbZhStOuOF387tjnXy2+v1Q+zDc1WqAvvvIzJgu7zuqHTlSZHnWW1VoW9aAJZcGPnQ3cSjuR3yIRUIIBfQ23Yh9Zr948Khj5xDQkB707eH56XYCPrJ06EoN5w7p3+hpvRq38q6FDCJWglSnR68b9FW684YR3vdwsXt1bwSv2yWWl9/HgwPk2Dih3OQmhrISQj6JSoYJVx05PMBGuctDKfMoyb8GG3bY7k+ggEyenX8k8LxMb3C2lyu6O9XyOTULgU3sYw1yCxXuxMxC//oqlV/a2K0af9e5zaZkt6LbKko0V2BniFCtMu4XijiDOr+r4Fe1Pn4GQm7nPhFYOvLAngpQgmczrmKLuwqoQcvYPOBUh0SkxrFzeWsaj28sFvebrvUS+urr6pg3OJuNBDxJhTFt9pc/9BQ9Sk3R7oiru21ieDRIf9l3ibQnjDih9HuilFznESica3bHia1/ePeUQkQ1Khg4GSpeTXp8bXis875usoXDD4Y9Dt6PBZ1G1a9ZyMfYJ1XuhV6lttVlWhCX2D65BN+4aaBRkUm1sM5bv1hMuRxSUuIUkr9T/U2LVzvb9FTg7Dm+YlHdXlhLRKQwPg0d/FOmqsczdtxCcR1xe3eXoYZVxLthos5R2ZMvBtCYwuFGH55lvnC+4NCFiIuv+agvQ0S8O/rm5UdIUpT1bIksvAWGw6bbFvk8wr6pR+qZlCZX3MYg71RzAkQlyLtG54JZQLd1XBsUostRYfz2i0xg8rO7TnTTu01UX7qqyvUFvdO/sTFTYctEFLI31ro4WDkOUpdVs2gHmS45pZO4qdC2EnoMY4sBJ5Ubn6yu7lez8dZDly3dQPENh0OoS5IQc8GtI1HC0AZg8iByHiKAkatn++zzyY4rQUePq79FwUD1za92d0+mn5S8/9V31Xp0cTlpickczeWralophrvNl8Rhei9fYXhOKRWPPu3TyDSEpfD8wAF1p3V7DeU1ZM+xEv3mH1d2WCN3PcMCkdKeOR80pBMxRXEhsUlpmK6j088VfAUXimT5XOjOrkHRo88W7mXaEqfQ3V5kpkxBWrDlf8mO5exVrMKm6P4LpUti7VyUo9wLH28ODXvNo9+cqjimxX15iobCc8x6pashzOmj4xlQ59l2Kkk+UvdTOtcKpi9lS/tQUrf2ZUs+ty6egVt6UZmHtzOgGoxJrDqOXe/hv6zB6+/LO8CrAIoKql81qQ6pWgdJUpzLupDwPFceRrF7CpnYUKi/W1UNPSkPdy932wyy40twrb9pRUC0JstkrDrkrZjK94EvV52JKoSkSZCIMm1HZppZdg/s0rrpskGM1HEZOhxwSKc6euA6FaIuVahKoc2sq4FrIcYlU0Ci6LxVG9NfBTSP7itxPK2/7SpQYlC8TvH84uKxGyU74Ylh9baxp5QS+KEoxaYuYPS4oEeO5ToI7WUHSVHgNkiNn5SH48KC/f9C0xdU4ZrJAo3jw+Q7sqHkLwHyd0Ylb3wqk7EHDCUy1IQ/1/VoG9+KEDPplQY3lHOpAk0bVaVi1rLi/XHFbPHp+0j95/qjpZDLMltipNMZSGCdDNzb/7QSXTaE8n3LxOeNYVifQfB1le4+WI/79MI6WUSCtNMC+juTe4ZLJ1j+JBvXzaeP5/tLzlYb3hiGR+vVX/eRfOpPSj1kjJ+woBF5kKsif+AuzSwfR7Rj2tXRJ+OQ1oG9RQMrkUvfCapAl+qZDSGtBu+OLMoeELLRlBqWbH59po1A1Ju1s4FIfjqJ5Y8tEJ88C1jQaRhg+JZtHfuCdgPV11oOS9kj7FM85rebM2X+67cJw049Ol1CB+SU7CpjLj/OnN9oJ4ni04ZcL3ZczqI+lBx3d3yzArYb7nsk3vh0sFkU5cKpR9nTzmHXfEqFLsO47Gx2Uaj3YLWFpEChJpTOpKruusjHbPtrkbbOTZ5UNYiq3mnO+Nq+X11bZpOzHKdK2eIzjTDy2frrYjEmd+6QqqsAG4txTdb4kmuDd49Slj8taOmV/lF4wVzH114LfXf6nZ3tAoT6ArQSdx9EYRdkWwKsusiiSKJp2gE1DLN50rMMT14Xznjr/f3Ku6fJ2KbfQ5Ko+MMCC5vsuLhV/d7UPh0q2SR69kOoQYVxZ4Jzd/Y9Goy7AOXnWgz70SB1UjTSNj2tcAzjhFqcEACN+oyRF992JOK6zg9bBzUor9aS7mlu2y9jnf9PYshq0Ow2jueUhZ12mdHlKBTG8VcfRejx5wvyAdXzJHgM3KbO9vZ+z1Rbbp3CzEpd68Z7DX7PUfUKONkyWlTGROnF+QpCdlDnl2+IQlfZ8++aHf/+Pat9tiyc/O+yJJ/uHeOvZk+eHbg8AUX791a5I8bCf1sUT0k7tLtHHlY+q9hbRkyq3qJwwud49UJVfYqOPRHm3AU1hSp8wQZ/zxfcc0pcvC8vRKWuSMppDjrmuy3kY5d7Z2TnBQIk+mBCKDCzFWKfq7Xfqu4pg0XTIEfVtlr8NqiiworrOv4z2KHbnv8ZiKW070qjhftx6Hekt1+Pr291syuF/0wtVXnQCeZSUQgsNaMe3nxd0vlX5k5UrsxSc9081cmMbF4bFQUMnwu8n0O1/yl9FZxPOc0hxmGKONALJ943IjndwJ4kLV1ugihyuuYHz2Y+ID_RESTRITO_DA_EQUIPE+Hze/XzFpfY8H5UXWJHw1XdsMpnkQMgaZ1rrNn+g+iS3G1kk2T67/oGw5Mqz53gRHPadIbt80fbp9b7bE+V16Hcd664+TolzjtBNskYw2nuxMnuNYKHrMLZ6zG9fjVgikn4jatEyRTbkdKAaWtS+8FD7n/Miq/tDWfF320JbzJy1o0F1HFSMaTKnUn4tfi9UJP6+P/Rzrh1yfLEpE3mA4Duu31LaaZ0faIOgLalZXDWtqqvI9ki7scEEX3DQNH2QYYw9aIGXiWGg4GgFFTIRLtUYenzNihblEsS7W6yOMuM3PK94L7xoDcc9VgZMifiQe/+jlDNVy+1ddkHg97OPf66BPBnnmnl2LW/4lLdzGZi63P4KILaGQRVexyBFEf7JWz/IrB9MSi6/Ojv+2xx1dvtVXXKMdRcFpE92r8zXXq00mrZxvNHRwdhOuy3dTMrjSwN5LpL4fXlGDrOz0vHcX33rCslxJ+Y3pESqxmJXjfv+u+yt9BrkOcfPZ5NkPNBpKL8girtwuIVpdRUIynTecsIER/ktHH4QsQaZtzI1G9cQ9qAJo/u8gWjyN+ID_RESTRITO_DA_EQUIPE/hbsxH8zN7gCs4rQOIFivtLpYYYz+APUtssbUJSJtoY9v5uER1r6qOdfPtxqfajnl2GeSmMk/0j8/sk3/wdQSwMEFAAAAAgAzbtQXbi+dQDQEwAA11EAAA0AAABzZW1lbnRlcy5qc29u1VzbbttImr6fpyj0VTdgnnSy1MZioNhO4p440druZAeDgVEiS1LFJItdRcpRBgvM1T7A7j7A9OzFoBfIzWb3Zi9Xb9JPsv9fRUmUSMpyzBlMXzgRT2JVff/h+w+l3/2KkD/AHyFfKcaCWx589S356sr1vjoyZ1OehgzPjd6eDf+RJNk45D5JZiIVcxGmFA+YnAgZ0dhnJKApJT//8d/Jm/Ozi9V3BILjN3iu3eoO+h3H63d6bmv9CpqlMyEV3oJPEYfod60uLxiVcC3OwjA/k8kQb56laaK+dRx8py0SFjNuCzl1VDaOuFJcxMrpdHv99VREwn18ze/0MZzx7FZ+UR+0iwedr/Tn3+cP3/FYL00g/CxicUqXf1n+hyDp8ic/5j5dvUMKs1pnNBCKRCzg+D+sy/ZjASU8VikNzfEJgUVl8fIvlJyekme/JR3bXX1jIrmQPF3At7byU3Mm+WRxO17crtCZ0FAxuPjPRzV4tmrwpL7PlCKpEKEiNA4IriOCsB5wCuu4A2QFcObbnNXjB2M35eksG9u+iJzXNH1Fx1fCv+NMOck8oD/cmuHtA/CJMO3AwmBB4IXiiGiJTjV2iRRzFvPlf8Y+p83C0i7Bci1CKslIKI4LT4bhFF8zi3BAxFy8ogGn5mqSgODoz0pr3eur81fOzcjq9lyr3WlvYK9H7ooF9Ihc2Br9YRxIRtURGdo7ELZct1+jfsJXdhxKeyrm+sCZLNy+UCl39BDsJJjsQ9DbaF3riXDispk7lF4pRvzlj6GfhQCtIMs/x1P8+BHABNU7IZLNudK4C/gc0nT5GVBtFuFOCeEXYKgkDckO0qcUB2qwfBi112+GQ/IiFGP4oksB84GhxlMCCiQkiK1cHK6BUWjHglIN31TSwIG182Ewjl5D9kOs7NHZ8/0IPgW0IejXBx7ld2WKgnFUAh8E1aOSAjRwR5B95JII4gMky58AGlTWgKsEJj/mIQ+oPs5k3LCKdksATll6yz6kkt7KtSai8l2D7fOZBKsPMAUnJJmHfExc2+utZXwPouZuX8Sp5OMsxbO7GtjqVUOoH7WSBXxXbIP+BumMaa3kwmGxMzdDcCSbMIlDdKZaBFMWmEdtLvVM4JJdMTl7lkbhXw//C/1yUE6wrkS/OmVSMpVKdoLrAW4SRQHg1RMwZph47V7vyCPvnOj//gv1V4lJeg/i2iz4vRL4lyJgIapaQBchn85SQueUhxRlMF1oI7pZTRh+BAIKa6DIRIoIBFQyP9V3TY3ybm4usyXP9XqO67b7ljto/dPXA/ebget2u9bLKgcM0H48IlfGjl8AF/JnLD4iI3PimsFU15cv4SIc3y2OyHf59ZTB6qV4x47YeYOBW2f4Dd86ZKRlsWkVuVe7ToZgTHwqdkVGg6AtAEhLDDZgZfYDPgEDcgLOEkUGBGDKYxo2KxPHJZkYKgWkAYUCFI+ILA0EOGugQiDK+iyL9OdMwtXJNnmOwLKFbNfi6yV1W07CE7vvtSvwfk7BZAK+Z49x1AW8yl9eBZFXhKj1BRChrY64L8AVc4BEJpIVyJZimQIdYRPuc7T3qlmk+iWkLiJN5tAqabGfU/Qc2oIDMBTRYKGmWlsgUVDSRTHOqdHV9zb4TLStNqDQtd2e7XpuBXhnDLx/CizzndG+34SMA5bXNvAufeIZ8+80vO/sSipW4wh2NXLveMp4t4sRUNvubA46j9PPZ1QxMll+UmDuMdZBNw5eUfKPK+xRVQVysv8FlwDxD4+SkBWcxen5KeEBHHOQDc0IGH5JQDUvaFZMBiUxMb7YONRvQSzMJ5iGf0enTMtHtPIDOdOERZ4uiFooUPYKbW55ntt13gulbNDPfqdCKF6KMJpKZkB/buTgJVg3PHNqr0Tlkt+JibrjR+SyUjS8GuXXrwbzJN5rCUkoSLNy9oysyhx0iuLRfoxEXBVcd8FbnxDAS1NwuJ7yuYE3J4HbzKkRrAvSv8Iaqa6WT3XrM1+vsURRZXN2q2ikmd0vh8klcyOC9va0HqBwJcV/Ep+rUGXjixkQIIioYVjLP6M5gM/Xw0uIr5licq4zJnDLd9dvXjcLelUy6x1NgZS9BelD498llzTOgI4Vg+jesOVavVavcwDmZ2IsdNgMhKusk53DI2evoyNn/dr9kXPb7h5goA8E7VTEK008Pf35j/92OiSGvYII5TCekgTsXsFbr517s2hVpKoKCcaRcSUwVm2GX0geWDD4GGg1C8joLbk2JniD5OnI6gKQ7eN2t182zGD9Bs7o7fWpcZFeB0xgtyo1eUlhdmCKnxk7LEAzKxTb7T4iSdI1SRIc2ENQ97ahLhy0igc1NvnQ9AmG2WCFdTAN0jy6wuwXkwFVJEYAQnDgcNAs3uUc2M3VxRur5dqu0zq2e9bNK+v7a42nHiDET0ZZH9bK4bNnB+dBwHpKKhc2s+nYJCRNrtsZBMFkQju9LqVepzeGj+1x4Pa7HbjP9/uesx6utRnubT7UWzPUh+DdcqzdJ4H4ikccmLRW33jCp5lW4P+BM+B3udZyIU+2Lm7cL5K1DCIZSmI8iwYA+DJg3izk5aTYUPtLHmci04odIR/HaErHtRjXQWAHAcPdosDR4Q9IRJD7bJxZISCsyFtX0eIWhJ5eyx1UxVhnwAIgtDtasfLvmGFjcJgH0uuou+zha21+PUUvjeWBikUphC5YghohqmZnN9tRtJESTMYkIaYTMS0KQfW5HCsrXxOSI7M6blY+KnJuM36LGZTbRNDbwLzzFlf5F8bPipm2uik9SNaemG/b4Wdsk1MFM59F+MmHj7pGtSIFozfDn//lX1+8vGgW6HJ+7apKj0mIRq1Q53gc4AU9bhrpKQ1DJhfOBldra+QOaFB6K/EcnGK3hYn8lWE+YxHmTiXF+BWiu0KWnGHQmuVeQRt6HYZTdUImzJ/pIEzkF37KcyQryxCwhMVB40G4V87VMDA3K7345Wp5cRYPJtIPyYgeDL/OPa9MOgOmGWFQDeCDFmOcdfYa/iNnLy+OyJgqhhfhLNbB8keiZPlJFr66GaDL2ZbzDwn4UsylmJLYucmlvEkg3kizGPSIqW/Jc8yfXwkxgYXDgjV5lvFQP3UBwjiVBxaqv7evbXKGyaRUZwGBZpj3HUwU7+/vbZPu0WTejxh38ryPg+7VD5ly2GpOls4PWeYBSxTnZMl8NtY4n4rFy1Mpy0mtcz881kvZhxS1OdWMX5c9UTKeXYzenhjFB24FMRXb0n6QkAkF+wD/z3gi4PvTpsWjVU7Q5OJQDAKvdF7dRIDvYPLiHqKVMwEQH5lqxp2pzRxQSG1cHsxHRWHu+WerkDi2TElAWfdm2BaWCpQFg7bU7qCbR/4MYveYr9Pv1y9fYGpGOyzsmMHTWBH/NGfhCRHmOkQN2JryI4rIyiX4TOaJ2WZj/1aZ+BmTcIZtRTe6SSUAv6bdwXMKUYK5PMT8/MNYj96SYUzDBUitIu/YGOIceTBxT+ZWim6EwxzsvG0F/AE+cLv+1tv8W03t3MLOJEv31lg47Aft/04DUoHoF67sxP9PS9atbQF6i5wWSpSNHzKa0wVmKll5o1K8KtKLVXE+EgA0YRLTeTRMRbPpgVaZIq5alEzPEa6xYuljG5ae0KiUP+psvdEZh2LsRJTHpnXJjoJ9SD8NtXPgdVkKwXu5s2yqe0sCzd/xXzoGXW0ak3IJFBxzSiJGY6JAciTT8iDJ11eX1+ffEKyX4DU6RuKZrq9eDs+/+TVo71TPAAY7pdgWR+hccO3a8XFwUzoTEPI0L6KWw/quN+g40yiwji2v1Tm2MOdaAfrpjPIjcmNi9zNJPwB310XxigC+LmkLbwEpAI0AYDKlY/m11z928O0OPuzss+I16NdUUsHNIg1TWzrKlNE/ig7aYH+EN4Dubmw8BvDLTz5EHJr0Yy0cJGadyjM3xcANdZPbe5Y2bM7LpP5acF01eyWwZv6wml4ji6LktdYxGP26wQn4k24UHV2eHqzDyTyJfFvpr9S+elXEs6YZD5jjWfeMYl4XrLXi0xjoWJKlylFAeQyV04PX3hrIf8h8HJUV6rk4q6v54b462tPStJfMpzFXkelXTFmc5Q9gfl5l71dmu6LgDrRt+TllquFKe6tM6kchjXW3g/bN5OvRm+E35KKu2eXvFfkEZ2GJiaU7AKxEUGsT2+2BuD5N99QWx9NVRxHTGRpYVkzgZIi5duLwJyXLY/v3yx9BCuAMSAYm7snyT7onWb+iUQlol3n7LmEXf2eYdy3qWyZws0SWAt7I8YpcXbcs+CAFO/xd7DXuDRZkdvj6b7GjdXR1AtF8wnxNwNeBWbE3Dm7lhaa6JoG+rtgZsIrapzxmTLehgtrfgDKBl8GSDXZUA36jQkeN6TXH0K0qRe+2nMFx3/O8QbfjtjdepCApZdq+06S2TzcL/Hn/DoCC1nq1+tzeVu6tDE7hYCu3U9dns1XTfWrVL+RzWWITL5jQYk2PiLZjOXPQMeCWBPk0FChviqOGUAy21/BV8sn2l8tUueT7EuRjLMQdilJRcMg11p7BoqD8FESuXozczrE7OIZ/qghhmfV5h4tRq1Y+2n8z4AsHNamBSil4nsWmlQoiBNOnhd0Lup0a89ZzPEKXF7/XOyDytFSd23gC8nUbIPKkT9GobCyJTvHUdVrpqtopImm5lnuMPamtCuSvBZAjrrCW9xuQ9CmXInswBvjbGZUD7UjvEZhfMT+TKt8acQSBYm4IAF/tGanuz9ko/OOB3l5l4Jq3SmTSZ0WvrGYisRncPOdM6nBaZeP3wKSVg/oOzna2wLbB0FJG001yjG0EwSl81tdS5s9iARgu8kR8WHt7fv3XCUz2HzytMbWiWS5SG9FcF6Xo1t4c0we42RFAC3tzaqR0qz0TYG7ZrltVhy7JpfsouawUkepYc6jPYs4vnxwEGdejYZ7wicVmPw2KjwlMqyxC58stQjkF+MbE/8Uu6lUVyq9rqCYMozMMm8Ot/YlDvemH3eNjbGI6mfwFYHrPpKHOB2/BKaMyOBSVxzRWm40PZqNSWaVD6iNVloXV0UkhCPHgOJ4JWIgvgSmV2V6Uylk5IHy4ori9AdYWCWqKmSgajhmIDipEBHHIDPs7FATKPp4BO+7LBe6DBF7O1ueNKm2hGsAK+GxPC/ZBzr19OECPana/wmRAmm9hgdmiY5UxltYlQ5eaUkPZ03VuruBpG8WlnJm7iBJqMAkylZLq5S2qyBXIDpX+jMCwU+zPn2GNO57mjldiYRGoQ/DFeuIdTNVLee8v1JMi3WGmmcCEz4XsmXZ9efKkUUjKSbB3QobBPcSgRCQpj7KIpDzEDQnTcLXIuLhKK1IWm21GEMfrhniSQfyvH2Cm8wmEyr+DzzmsNAbvarYz8jnDQikAwT8KCDbD/OqXYNY/HLNHZTcvYh90f92KgqH+OiyOKUQhyTp9hZs0NYVpFJ5y6gpTLJLNWKxwAYdJIgUFdYCVXG/+0gwUkAvXe0e2ghSd86ry+thsejM6f4UOf2CjRer3qtjpE3zLYb3cezbtaB648ho6FIyWn8Gnat1ZBQWN+vyKrnzEIAOGDj6huMrp2slo5Sg4+YiriKb+DBSIx9vGzWdhaJKhvmkUxi/zufQzXqoQr/eytY8HVu+4f/x1v6/3sg0qo4gdnLxB/xF68gSc8smukrwwL/ZBK9AKtmbxqWigLziMy3VDJnZjk0I3dhGFC+wlSws908XNMccQqrfcTmdwkC4cH64LNUFxebf3zr41zH+AXcUY8ePGP6zaZxnWYnCSusS2/BMYpkc6jQeXvKILvriaZxiP5zvXrih6s2G8Lm6DOlxplluTFcGNf57XP8jwHM6ZnuCfR6bHHHmrnlZu/vVGftPHDA6cRTq/DYu//BRkSKIYQdcpeaQ3Cze7/uXExAO7EEjVLoQaF3DIfoOy2HcPF/ut/NEhucJqO1O9IaCwfScohhrNAlAOv9+ZEg0stJT5Qj9QYljZFs897rvd46blvWYt643L+YRxoBEoOQLOLT9PcWssBAqjq2YXrxxED4M5BWKFLD73o1UV+cvhOdHtHHiDrK7pr2rydL3rGXudcF+k6czfu0223e52HF+6bbDihxn7g6X+kWX1i53q6G4DDCzFkZkrI3O+/KlhylMOoNf9jKsmQBDw7SDNbCkBU69DM12XBJwsmN8qq4G8dJLpFIlchXFbPYf7MlFAKZCSejoRtRngXm1pHY7O3h0J+7zDJf4gBKccDBD4gby0qX8YQqJxyh3HOuGBTYzNYlWx45/csUUhlTR+CDscVP3u4ByD9+DH/BlicAwA2K57kIZ4B9OhJ2BwUagAQBDwA1Bnli7/G4uWR4VgYb0nsGECWo6hn8FakRFSNO6DOKzrReiO0XidijDvz9BK8b3SmcL8B5SYyUubtsLNDyvlRYetX1W6mXEZkPOgZj+RcS/HECi4VRv9S2C1vqy+VF9EqO0T30NxVx0AR1tdfvgzO6BFup9tnu8UWLeRmd8CatgGVraCM+RzOlmBbFuBYG5Ff1tZYK5Mcz9eR49k1GwCYlpI4hTVrOs5LHmfzHXbpts+iAHvNoR+sX6V2MAHmEuou7Q2kwZmw/Kfu9lK6Ubr1uvG7VtFz/U1i7hldoUB7HFabrbfDe+MBr0I6UdzVwE6zTjON6E66uObxMQpBQZXbOfe1kGA9kbiauvAZlq9sc/rdrHxbuD03EGv0z+IUnsHZ1UeCe1ZRf4dVGz5GaetDS1oGBLoraI6BDsRTzerXqlvvYdx/tXv/x9QSwMEFAAAAAgAzbtQXRuoaOEKAgAAfQMAAA4AAABwcm90b2NvbG8uanNvbm1T227TQBB971es/JxEtpUGkj5VyKIRgaSUSwpCq4l3nK5q77p7idIiPgZ4QDwjvsA/xnhthRb6Nj5zOTN7jj8fMRbt0FipVTRjUTKKR3E0aFGb6xpbrPBKQIXKQfOj+a6Zw+aXkTmcMIsV23ibAxPISsi9AqY9gxK3fS3hBnfStrGV1mHVfHXU21EURlf8FsEQTTI9jgPoroz22ysuwAX6NE4nwyQeJpOuyWBOu/D7vVSShBxtrIh9z2s0/MajuW2zxw9z/YD7JUkaSnKjrTVYPGzv1pLKOum8o4eC8rH5BgmwjjtZofaOW8y1EvZvQQV7WfmKg6NnqF3I9LQoaCMJZYt9JIjA5Sp7dbrI1vx0NecvsstwPOEXz5artxf/ou+X/0HzLMsOGEGfuiW0wJIr7XCj9TUXRu6QSxG0v7xb7tfvnmzOdXFzPnk+n47VWbJIzHJ4tlin4zR6bAJ9ykKiCHcf9IqnwyR9E49ncTJLxqN0+vRDrx6oa6m2beFrLHHXfFO5BHYwxoDpjQHbGqdqfjot9ICVckfCtNAGLA5YLoMVf6MlB9ZgQFBDrU1IM2S5ruoSW8dq1omNUa+hQ1MbdOB6v78EacM8QQRWbhXdQnamFtM61/qyS9EYT5qHGFkhSxfiASPH96zkV2BSCeqnfU6Yaj3fn8sq+odIX4Z78OSjnR5FR1/+AFBLAQIUAxQAAAAIAM27UF3XjT07CSYAAIF6AAASAAAAAAAAAAAAAACAAQAAAABiaWJsaW9ncmFmaWFfZnYucHlQSwECFAMUAAAACADNu1Bd4YU0mdMMAACSIgAAEgAAAAAAAAAAAAAAgAE5JgAAZm9ybWF0YWNhb19hYm50LnB5UEsBAhQDFAAAAAgAzbtQXVSvmxoOCgAACBgAABgAAAAAAAAAAAAAAIABPDMAAHdvcmRfcmVmZXJlbmNpYXNfYWJudC5weVBLAQIUAxQAAAAIAM27UF1DTXnlkhoAAJtZAAAKAAAAAAAAAAAAAACAAYA9AAB0ZW1hcy5qc29uUEsBAhQDFAAAAAgAzbtQXbi+dQDQEwAA11EAAA0AAAAAAAAAAAAAAIABOlgAAHNlbWVudGVzLmpzb25QSwECFAMUAAAACADNu1BdG6ho4QoCAAB9AwAADgAAAAAAAAAAAAAAgAE1bAAAcHJvdG9jb2xvLmpzb25QSwUGAAAAAAYABgB1AQAAa24AAAAA")
assert hashlib.sha256(support).hexdigest()=="cf0cbb4e6af487a31c5cc9781f653874a6951b1f218ee3248d7c45f6cf6bae05"
with zipfile.ZipFile(io.BytesIO(support)) as archive:
    for item in archive.infolist():
        target=(BIB/item.filename).resolve();assert target.is_relative_to(BIB.resolve())
        if not target.exists():target.write_bytes(archive.read(item.filename))
sys.path.insert(0,str(BIB))
import bibliografia_fv as bibliography
import importlib;importlib.reload(bibliography)
CONSULTAR_REDE=True
NOVA_COLETA=False
SURVEY=bibliography.Survey(BIB,live=CONSULTAR_REDE)
if NOVA_COLETA:
    SURVEY.raw=BIB/"raw"/("consulta_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"))
    SURVEY.raw.mkdir(parents=True,exist_ok=True)
    SURVEY.cfg["through_date"]=date.today().isoformat()
print("Saídas:",SURVEY.out)
print("Chaves presentes:",{provider:bool(value) for provider,value in SURVEY.keys.items()})


## 1 Protocolo e consultas dos 19 tópicos

<details>
<summary>Descrição e método</summary>

As consultas do protocolo preservam os tópicos do plano. Obras de método e livros entram como sementes explícitas. A seleção automática exige termos específicos no título, além da pertinência temática. Isso evita promover artigos de células ou física solar a referências sobre inversores ou variabilidade do recurso terrestre somente por terem muitas citações.

</details>

In [ ]:
display(pd.DataFrame(SURVEY.topics)[["id","title","owner","query"]].rename(columns={"id":"Subseção","title":"Tópico","owner":"Responsável","query":"Consulta"}))
print(SURVEY.cfg["ranking"])


## 2 Coleta multibase com rastreabilidade

<details>
<summary>Descrição e método</summary>

As respostas originais ficam em raw e as consultas efetivas em outputs/logs. O retrato padrão é limitado ao protocolo de 16/10/2026. Consultas são limitadas por tópico, sem alegação de exaustividade. A chave pode não conferir acesso ao conteúdo integral ou a todos os campos de uma base. Falhas e ausência de credenciais não são representadas como buscas bem-sucedidas ou zero publicações.

</details>

In [ ]:
REGISTROS=SURVEY.collect()
display(SURVEY.availability)
display(pd.DataFrame(REGISTROS).groupby("source").size().rename("Registros"))
print("Registros normalizados:",len(REGISTROS))


## 3 Deduplicação classificação e seleção de leituras

<details>
<summary>Descrição e método</summary>

DOI normalizado é a primeira chave. Sem DOI, usa-se título normalizado e ano, preservando edições distintas. As citações de bases diferentes não são somadas. A guia combina livro, métodos específicos, trabalhos citados no conjunto recuperado e complemento recente. Leitura integral e conferência de páginas ou equações continuam necessárias antes da redação.

</details>

In [ ]:
OBRAS=SURVEY.consolidate()
LEITURAS=SURVEY.select()
RESUMO=SURVEY.export()
display(pd.DataFrame([RESUMO]).drop(columns=["source_status"]))
display(pd.DataFrame(LEITURAS).groupby("topic").size().rename("Leituras indicadas"))


# 1 Recurso solar

<details>
<summary>Descrição e método</summary>

Referências para a seção 1. A lista completa, o ranking recuperado e os DOI estão nos CSV e documentos por seção.

</details>

## 1.1 Geometria solar e irradiância extraterrestre

<details>
<summary>Descrição e método</summary>

Responsável: Samuel. Calcular a posição solar com SPA e a irradiância extraterrestre com Spencer. O azimute é contado a partir do norte no sentido horário.

Limite do estudo de caso: Não há observação independente de ângulo solar; a geometria pode ser calculada.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="1.1"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 1.2 Grandezas radiométricas e componentes da radiação

<details>
<summary>Descrição e método</summary>

Responsável: Samuel. Irradiância é potência por área; irradiação é sua integral no tempo. GHI é horizontal, DNI é normal aos raios solares e DHI é difusa horizontal. POA se refere ao plano do arranjo.

Limite do estudo de caso: Sem GHI/DNI/DHI locais medidos; reconstrução não equivale a observação.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="1.2"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 1.3 Modulação atmosférica e variabilidade espaço temporal

<details>
<summary>Descrição e método</summary>

Responsável: Samuel. Nuvens, aerossóis e vapor d’água modificam o recurso solar. A série permite descrever a variabilidade intradiária e entre dias; a seleção dos dias ilustrativos usa apenas POA e cobertura.

Limite do estudo de caso: Não se atribuem causas atmosféricas isoladas nem climatologia anual a um trimestre.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="1.3"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 1.4 Medição e estimativa do recurso solar

<details>
<summary>Descrição e método</summary>

Responsável: Samuel. Inspecionar integridade, unidades, lacunas, simultaneidade e limites físicos. Medições de superfície, satélites, reanálises e atlas têm representatividades diferentes; os produtos em grade são contextualizados, sem uso na cadeia principal.

Limite do estudo de caso: Disponibilidade elevada não certifica sensores. Altura do anemômetro e alguns metadados permanecem incompletos.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="1.4"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


# 2 Ambiente e montagem

<details>
<summary>Descrição e método</summary>

Referências para a seção 2. A lista completa, o ranking recuperado e os DOI estão nos CSV e documentos por seção.

</details>

## 2.1 Condições térmicas

<details>
<summary>Descrição e método</summary>

Responsável: Vannessa e Thais. Temperatura ambiente, vento e trocas de calor condicionam a operação do módulo. As distribuições observadas e o cenário de menor ventilação ilustram a passagem do ambiente à modelagem térmica da seção 3.2.

Limite do estudo de caso: U₀ e U₁ genéricos; T operacional é aproximação de célula, sem validação térmica independente.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="2.1"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 2.2 Albedo e ambiente radiativo da superfície

<details>
<summary>Descrição e método</summary>

Responsável: Vannessa e Thais. O albedo expressa refletância do entorno. No exercício de transposição, varia-se 0,10/0,20/0,40 mantendo as componentes reconstruídas na referência 0,20. O arranjo real é monofacial; bifacialidade exige caracterização traseira adicional.

Limite do estudo de caso: Expressão isotrópica para reflexão do solo. Não há albedo medido; a leitura POA principal não recebe uma segunda contribuição refletida.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="2.2"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 2.3 Sombreamento sujidade e obstruções locais

<details>
<summary>Descrição e método</summary>

Responsável: Vannessa e Thais. A sujeira pode atenuar a luz incidente e diferir entre módulos e sensor POA. Os cenários são hipóteses de atenuação adicional à leitura. Sombreamento parcial pode envolver mismatch e diodos de bypass, que um redutor uniforme não reproduz.

Limite do estudo de caso: Sem histórico de limpeza e mapa de sombras; o resíduo não identifica a perda real de sujidade ou sombra.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="2.3"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 2.4 Geometria de instalação e exposição do arranjo fotovoltaico

<details>
<summary>Descrição e método</summary>

Responsável: Vannessa e Thais. A instalação define exposição e incidência. A inclinação é medida em relação à horizontal. Espaçamento entre fileiras e exposição traseira precisam de geometria adicional; em um exemplo de sombra coplanar, uma altura h gera projeção horizontal h/tan(altura solar).

Limite do estudo de caso: A expressão de projeção é um exemplo plano sem dimensão de fileira real; não calcula o sombreamento específico da usina.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="2.4"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


# 3 Conversão fotovoltaica

<details>
<summary>Descrição e método</summary>

Referências para a seção 3. A lista completa, o ranking recuperado e os DOI estão nos CSV e documentos por seção.

</details>

## 3.1 Transposição para o plano do arranjo

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. A cadeia usual transpõe GHI/DNI/DHI para POA. Como a instalação já mede POA, o modelo principal começa dessa observação. A inversão opcional usa Driesse 2024, Erbs–Driesse e Perez–Driesse para exercícios de orientação/albedo.

Limite do estudo de caso: Há possibilidade de soluções não únicas; o fechamento de POA é teste numérico, sem validação independente das componentes.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.1"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.2 Modelagem da temperatura operacional

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. A temperatura estimada entra no modelo elétrico. U₀=25 W/(m² K) e U₁=6,84 W s/(m³ K) são parâmetros genéricos de referência; a sensibilidade de ventilação usa 15 e 3, respectivamente.

Limite do estudo de caso: Ausência de temperatura de módulo/célula e altura de vento suficientemente documentada.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.2"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.3 Modelagem elétrica do módulo fotovoltaico

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. O modelo de diodo único fornece curvas I–V, P–V e o ponto de máxima potência. Compare irradiâncias a 25 °C e temperaturas a 1.000 W/m². Registre divergências de classificação e parâmetros entre CEC e metadados.

Limite do estudo de caso: Sem varreduras I–V experimentais da instalação; as curvas não constituem validação do módulo.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.3"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.4 Modelagem do arranjo fotovoltaico em corrente contínua

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. Em uma associação ideal, as tensões somam em série e as correntes em paralelo. O exemplo 20 módulos × 6 strings por grupo corresponde a 120 módulos, mas não confirma o cabeamento real ou a ligação aos MPPTs.

Limite do estudo de caso: Strings/MPPTs reais incompletos; V×I disponível não foi adotado como potência DC observada certificada.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.4"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.5 Conversão de CC para CA e modelagem do inversor

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. A curva PVWatts estima conversão e clipping. O limite de referência é 30 kW por inversor; 27,6 kW permanece como sensibilidade documentada. Verificar a faixa real de MPPT exige topologia e configuração de campo adicionais.

Limite do estudo de caso: A simulação agregada não verifica todas as restrições reais de tensão/MPPT nem confirma o limite instalado.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.5"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 3.6 Integração temporal e estimativa de energia

<details>
<summary>Descrição e método</summary>

Responsável: Gabriela e Giulia. Integre potência média por duração de cada intervalo. Compare séries e energias nas mesmas janelas. Apresente energia mensal e trimestral observada completa separadamente dos intervalos diurnos pareados.

Limite do estudo de caso: O recorte é trimestral, não uma medição anual; energia pareada não equivale ao total integral do dia.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="3.6"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


# 4 Desempenho e validação

<details>
<summary>Descrição e método</summary>

Referências para a seção 4. A lista completa, o ranking recuperado e os DOI estão nos CSV e documentos por seção.

</details>

## 4.1 Balanço energético e cadeia de perdas

<details>
<summary>Descrição e método</summary>

Responsável: Júlia Ribeiro. Integre a cadeia óptica, térmica, elétrica DC, conversão e limite AC. A diferença entre estágios pertence ao modelo; um desvio do medidor pode envolver fenômenos e incertezas não identificados.

Limite do estudo de caso: Não se identificam perdas ópticas/operacionais reais isoladas sem medições adicionais; ganhos sob frio podem ocorrer.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.1"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 4.2 Produtividades de referência do arranjo e final

<details>
<summary>Descrição e método</summary>

Responsável: Lucas Simão. Yr é o recurso em horas equivalentes; Ya representa a energia DC por capacidade; Yf representa energia AC por capacidade. HSP/PSH corresponde ao mesmo conceito de recurso quando plano e janela são os mesmos.

Limite do estudo de caso: Ya é modelada; HSP da janela pareada não é duração de brilho solar nem total integral diário.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.2"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 4.3 Razão de desempenho e decomposição normalizada das perdas

<details>
<summary>Descrição e método</summary>

Responsável: Lucas Simão. Compare PR observado e modelado com irradiação e energia da mesma janela. Lc e Ls decompõem o modelo em perdas de captura e sistema. Informe a normalização e o papel da potência declarada.

Limite do estudo de caso: Não combinar energia integral do trimestre com POA parcial; perdas calculadas não são identificação experimental de cada causa.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.3"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 4.4 Validação de modelos de geração fotovoltaica

<details>
<summary>Descrição e método</summary>

Responsável: Lucas Simão. Avalie viés, magnitude do erro, correlação e determinação. Erro=simulado−observado. MBE, MAE e RMSE são em kW; nMBE/nRMSE são divididos pela média observada dos respectivos pares. O modelo físico é separado do fator empírico ajustado somente em abril.

Limite do estudo de caso: Junho é teste temporal separado, não instalação independente. Correlação elevada não prova exatidão; nenhum limiar universal de aceitação foi adotado.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.4"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


## 4.5 Avaliação de desempenho de sistemas integrados à edificação

<details>
<summary>Descrição e método</summary>

Responsável: Júlia Ribeiro. Explore fachadas sul/leste/oeste e menor ventilação. O exemplo separado de 100 m² semitransparentes usa transmissão solar 0,25, reflexão 0,12, absorção 0,63 e transmissão visível 0,35. As propriedades não caracterizam o módulo real.

Limite do estudo de caso: Somente horas elegíveis da inversão; sem fachada medida, SHGC certificado, cálculo de lux ou balanço térmico completo do edifício.

</details>

In [ ]:
for work in [w for w in SURVEY.selected if w["topic"]=="4.5"]:
    display(Markdown("**"+work["id"]+" — "+work["selection_role"]+"**\n\n"+bibliography.citation(work)+
        "\n\nDOI: "+("https://doi.org/"+work["doi"] if work["doi"] else "não informado")+
        "\n\n"+bibliography.counts(work)+"\n\nAcesso: "+(work["oa_url"] or work["url"])))


# 5 Documentos exportações e execução

<details>
<summary>Descrição e método</summary>

Word por subseção; Markdown e CSV por seção; catálogo consolidado e relações por tópico; BibTeX e RIS para gerenciadores de referências. Os formatos bibliográficos não importam automaticamente no Zotero nem publicam no GitHub. O grupo pode versionar o código e os resultados depois.

</details>

In [ ]:
print("Word:",SURVEY.out/"Referencias_por_Subsecao.docx")
print("BibTeX:",SURVEY.out/"referencias_fundamentacao.bib")
print("RIS:",SURVEY.out/"referencias_fundamentacao.ris")
display(pd.DataFrame([{"Arquivo":str(p.relative_to(SURVEY.out)),"Bytes":p.stat().st_size} for p in sorted(SURVEY.out.rglob("*")) if p.is_file()]))
assert set(w["topic"] for w in SURVEY.selected)==set(t["id"] for t in SURVEY.topics)
assert all(w["title"] and (w["doi"] or w["url"]) for w in SURVEY.selected)
assert not any("Solar Dynamics Observatory" in w["title"] or "polymer tandem" in w["title"].lower() for w in SURVEY.selected)
print("Conferência: 19 tópicos com referências efetivas e origem identificada.")
